# Combined Equation Model Exploration with Operator Composition

This notebook allows you to:
- Explore your DISCO model trained on combined physics equations (EULER, HEAT, DISP)
- Test operator composition methods (Greedy, Random, Exhaustive)
- Compare performance on out-of-distribution data
- Analyze operator usage patterns across physics types

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader
import random
from itertools import permutations
import math
import time

# Add project root to path
sys.path.append('/mnt/home/lserrano/disco-ball')
sys.path.append('/mnt/home/lserrano/disco-ball/tests/neural-operator-splitting')

#from train.train_combined_coda import DISCOLitModule# HDF5TemporalDataset
from train.train_combined_aggregate import DISCOLitModule# HDF5TemporalDataset
from src.utils.database import RelativeL2
from src.operators.disco import DISCOHouse
from operator_utils import sequential_operator_composition, strang_splitting_composition
from einops import rearrange

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
class HDF5TemporalDataset(torch.utils.data.Dataset):
    """Dataset for loading pre-computed trajectory data from HDF5 files"""
    
    def __init__(self, hdf5_files, input_frames=16, output_frames=16, 
                 sub_x=1, sub_t=1, split='train'):
        """
        Args:
            hdf5_files: List of HDF5 file paths to load data from
            input_frames: Number of input time frames
            output_frames: Number of output time frames  
            sub_x: Spatial subsampling factor
            sub_t: Temporal subsampling factor
            split: Dataset split ('train', 'val', 'test')
        """
        self.hdf5_files = hdf5_files if isinstance(hdf5_files, list) else [hdf5_files]
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        
        # Build file index for efficient access
        print("Building file index...")
        start_time = time.time()
        self.total_samples = self._build_file_index()
        index_time = time.time() - start_time
        print(f"Dataset length calculation took {index_time:.2f}s for {self.total_samples} samples")
        
        # Track loading times for performance assessment
        self.loading_times = []
        
    def _build_file_index(self):
        """Pre-compute file offsets for efficient __len__ and __getitem__"""
        self.file_offsets = []
        total_samples = 0
        
        for file_path in self.hdf5_files:
            if not os.path.exists(file_path):
                print(f"Warning: HDF5 file not found: {file_path}")
                continue
                
            try:
                with h5py.File(file_path, 'r') as f:
                    # Try different group names based on split
                    data_group = None
                    dataset_path = None
                    
                    # Check for split-specific groups first, then fall back to 'train'
                    possible_groups = [self.split, 'train', 'valid', 'test']
                    for group_name in possible_groups:
                        if group_name in f and 'pde_250-256' in f[group_name]:
                            data_group = group_name
                            dataset_path = f'{group_name}/pde_250-256'
                            break
                    
                    if data_group is None:
                        print(f"Warning: No valid dataset structure found in {file_path}. Checked groups: {possible_groups}")
                        continue
                        
                    n_samples = f[dataset_path].shape[0]
                    n_timesteps = f[dataset_path].shape[1]
                    
                    # Verify we have enough timesteps for input + output frames
                    min_timesteps_needed = (self.input_frames + self.output_frames) * self.sub_t
                    if n_timesteps < min_timesteps_needed:
                        print(f"Warning: Not enough timesteps in {file_path}. "
                              f"Need {min_timesteps_needed}, got {n_timesteps}")
                        continue
                    
                    self.file_offsets.append((file_path, total_samples, n_samples, dataset_path))
                    total_samples += n_samples
                    print(f"Added {n_samples} samples from {file_path} (using {dataset_path})")
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue
                
        if total_samples == 0:
            raise ValueError("No valid samples found in any HDF5 files!")
            
        return total_samples
    
    def _get_file_and_local_idx(self, idx):
        """Convert global index to file path and local index"""
        for file_path, offset, n_samples, dataset_path in self.file_offsets:
            if idx < offset + n_samples:
                local_idx = idx - offset
                return file_path, local_idx, dataset_path
        raise IndexError(f"Index {idx} out of range for dataset size {self.total_samples}")
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        start_time = time.time()
        
        # Find which file and local index
        file_path, local_idx, dataset_path = self._get_file_and_local_idx(idx)
        
        try:
            with h5py.File(file_path, 'r') as f:
                # Get the group containing the data
                group_name = dataset_path.split('/')[0]
                
                # Load trajectory data - shape: (n_timesteps, n_spatial)
                trajectory = f[dataset_path][local_idx]
                
                # Load PDE parameters (alpha, beta, gamma) for this sample
                alpha = f[group_name]['alpha'][local_idx]
                beta = f[group_name]['beta'][local_idx]
                gamma = f[group_name]['gamma'][local_idx]
                
                # Sample temporal window randomly
                total_frames_needed = self.input_frames + self.output_frames
                max_start = (trajectory.shape[0] // self.sub_t) - total_frames_needed
                if max_start <= 0:
                    # If not enough frames, use what we have
                    start_idx = 0
                    available_frames = trajectory.shape[0] // self.sub_t
                    actual_input_frames = min(self.input_frames, available_frames // 2)
                    actual_output_frames = available_frames - actual_input_frames
                else:
                    start_idx = np.random.randint(0, max_start + 1)
                    actual_input_frames = self.input_frames
                    actual_output_frames = self.output_frames
                
                # Apply temporal subsampling and extract sequences
                #start_t = 0
                start_t = 0 #start_index # 50
                input_end_t = start_t + actual_input_frames * self.sub_t
                output_end_t = input_end_t + actual_output_frames * self.sub_t
                
                input_seq = trajectory[start_t:input_end_t:self.sub_t, ::self.sub_x]
                output_seq = trajectory[input_end_t:output_end_t:self.sub_t, ::self.sub_x]
                
                # Add channel dimension and convert to torch tensors
                # Expected format: (time, channels, spatial)
                input_tensor = torch.from_numpy(input_seq).unsqueeze(-2).float()
                output_tensor = torch.from_numpy(output_seq).unsqueeze(-2).float()
                
                # Track loading time
                loading_time = time.time() - start_time
                if len(self.loading_times) < 1000:  # Collect first 1000 samples
                    self.loading_times.append(loading_time)
                
                return {
                    'input': input_tensor, 
                    'target': output_tensor,
                    'alpha': float(alpha),
                    'beta': float(beta),
                    'gamma': float(gamma),
                    'index': idx
                }
                
        except Exception as e:
            print(f"Error loading sample {idx} from {file_path}: {e}")
            # Return dummy data to avoid training crash
            dummy_input = torch.zeros(self.input_frames, 1, 256 // self.sub_x)
            dummy_output = torch.zeros(self.output_frames, 1, 256 // self.sub_x)
            return {'input': dummy_input, 'target': dummy_output}
    
    def get_loading_stats(self):
        """Return loading performance statistics"""
        if not self.loading_times:
            return {}
        
        return {
            'avg_loading_time': np.mean(self.loading_times),
            'min_loading_time': np.min(self.loading_times),
            'max_loading_time': np.max(self.loading_times),
            'samples_per_second': 1.0 / np.mean(self.loading_times),
            'total_samples_timed': len(self.loading_times)
        }

In [ ]:
# Model configuration - UPDATE THESE PATHS
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0.001_inframes16_outframes2_subx1_subt1"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1"
#run_name="DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_resumed"


#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_20250902_140828"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t128_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_20250904_002223"

# with coda
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_20250904_113524"

# with aggregate
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250908_234201"

# with aggregate 2
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250913_004759"


# test another one with high loss


run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250916_112046"


MODEL_CHECKPOINT_PATH = f"/mnt/home/lserrano/disco-ball/outputs/{run_name}/last.ckpt"
DATA_DIR = "/mnt/home/lserrano/disco-ball/datasets/combined_equation/"

VALIDATION_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_valid.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_valid.h5",
    'DISP': f"{DATA_DIR}E_DISP_valid.h5"
}

TEST_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_test.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_test.h5",
    'DISP': f"{DATA_DIR}E_DISP_test.h5"
}

# Training files for operator encoding
TRAINING_FILES = {
    #'EULER': f"{DATA_DIR}E_EULER_train_8192.h5",
    #'HEAT': f"{DATA_DIR}E_HEAT_train_8192.h5",
    #'DISP': f"{DATA_DIR}E_DISP_train_8192.h5"
    'EULER': f"{DATA_DIR}E_EULER_train_envsize64.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_train_envsize64.h5",
    'DISP': f"{DATA_DIR}E_DISP_train_envsize64.h5"
}

# Test configuration
BATCH_SIZE = 64
N_INPUT_FRAMES = 16
N_OUTPUT_FRAMES = 100
SUB_X = 1
SUB_T = 1 #1

# Operator composition configuration
OPERATOR_CONFIG = {
    'num_operators': 64,  # Number of operators to encode
    'n_trajectories_per_operator': 1,  # Trajectories per operator (anti-forgetting)
    'max_operators': 5,  # Maximum operators in composition
    'min_improvement_threshold': 5.0,  # Minimum improvement % to add operator
    'n_input_frames': N_INPUT_FRAMES,
    'n_output_frames': N_OUTPUT_FRAMES
}

print("Configuration:")
print(f"  Model path: {MODEL_CHECKPOINT_PATH}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Operator config: {OPERATOR_CONFIG}")

print("\nData files:")
for name, path in VALIDATION_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Validation {name}: {exists}")

for name, path in TRAINING_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Training {name}: {exists}")

## Load Model and Data

In [ ]:
def load_model_from_checkpoint(checkpoint_path):
    """Load DISCO model from Lightning checkpoint"""
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Please update MODEL_CHECKPOINT_PATH with your actual model path")
        return None, None
    
    try:
        lit_model = DISCOLitModule.load_from_checkpoint(checkpoint_path, map_location=device)
        lit_model.eval()
        
        model = lit_model.model.to(device)
        model.eval()
        
        print(f"Model loaded successfully from {checkpoint_path}")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        return model, lit_model
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# Load the model
model, lit_model = load_model_from_checkpoint(MODEL_CHECKPOINT_PATH)
relative_l2_error = RelativeL2()

if model is None:
    print("\nTo use this notebook, you need to:")
    print("1. Train a model using train_combined.py")
    print("2. Update MODEL_CHECKPOINT_PATH above with your checkpoint path")

In [ ]:
def create_dataset_for_equation(equation_type, split='val', files_dict=None):
    """Create dataset for a specific equation type"""
    files_dict = files_dict or VALIDATION_FILES
    
    if equation_type not in files_dict:
        print(f"Equation type {equation_type} not found in files dict")
        return None, None
        
    file_path = files_dict[equation_type]
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None, None
    
    dataset = HDF5TemporalDataset(
        hdf5_files=[file_path],
        input_frames=N_INPUT_FRAMES,
        output_frames=N_OUTPUT_FRAMES,
        sub_x=SUB_X,
        sub_t=SUB_T,
        split=split
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    print(f"{equation_type} {split} dataset: {len(dataset)} samples")
    return dataloader, dataset

# Create validation datasets
val_dataloaders = {}
val_datasets = {}

for eq_type in VALIDATION_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'val')
    if loader is not None:
        val_dataloaders[eq_type] = loader
        val_datasets[eq_type] = ds

print(f"\nLoaded {len(val_dataloaders)} validation datasets")


test_dataloaders = {}
test_datasets = {}

for eq_type in TEST_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'test', files_dict=TEST_FILES)
    if loader is not None:
        test_dataloaders[eq_type] = loader
        test_datasets[eq_type] = ds


train_dataloaders = {}
train_datasets = {}

for eq_type in TRAINING_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'train', files_dict=TRAINING_FILES)
    if loader is not None:
        train_dataloaders[eq_type] = loader
        train_datasets[eq_type] = ds

In [ ]:
eq_type="HEAT"
num_integration_steps=1
N_OUTPUT_FRAMES=100

total_error = 0
test_size = 0
all_theta_latent = []
all_alpha = []
all_beta = []
all_gamma = []
all_errors = []

for batch in tqdm(train_dataloaders[eq_type]):
    inp, target = batch["input"], batch["target"]
    alpha, beta, gamma = batch['alpha'], batch['beta'], batch['gamma']
    
    all_alpha.append(alpha)
    all_beta.append(beta)
    all_gamma.append(gamma)
    
    inp = inp.squeeze(1)
    target = target.squeeze(1)
            
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
        
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
        #theta_latent = lit_model.codes[batch['index']]
        #theta_latent = lit_model.codes[[random.randint(0, 1023) for j in range(B)]]
        #theta_latent = lit_model.codes[]
        all_theta_latent.append(theta_latent.cpu())
        theta = model.decode_theta(theta_latent, dim)
        n_output_frames = N_OUTPUT_FRAMES
        #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames,)# integration_time=4/250*n_output_frames, dt=4/250, predict_normed=False, metadata=metadata)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, integration_time=4/250, dt=1/256)#dt=4/250, predict_normed=False, metadata=metadata)

    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    # new
    sample_rollout_error = relative_l2_error(pred, target[:, :n_output_frames], ).item()
    x = rearrange(pred.clone(), "b ... -> b (...)")
    y = rearrange(target[:, :n_output_frames].clone(), "b ... -> b (...)")
    diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
    y_norms = torch.linalg.norm(y, ord=2, dim=-1)
    sample_rollout_error = diff_norms / y_norms

    all_errors.append(sample_rollout_error)
    
    total_error+=rollout_error*n_sample
    test_size+=n_sample
    
all_errors = torch.cat(all_errors)
print('test error', total_error/test_size)

In [ ]:
print(pred.shape, target.shape)

In [ ]:
#print(f"Error with encoder", rollout_error)
pred.shape

In [ ]:
idx=10
for t in range(10):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
idx=10
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze().cpu().detach()[t])

In [ ]:
def plot_scatter(all_alpha, all_errors, run_name, title="train_t1"):
    plt.figure(figsize=(8, 6))
    plt.scatter(all_alpha, all_errors.cpu().detach(), alpha=0.6, s=30)
    plt.xlabel("Alpha"), plt.ylabel("Error"), plt.title(title), plt.grid(alpha=0.3)
    plt.yscale('log')  # Log scale for y-axis
    Path(f"plots/{run_name}").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"plots/{run_name}/{title}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_scatter(all_alpha, all_errors, run_name, title="train_t50")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

def plot_3d_scatter(all_theta_latent, all_alpha, run_name, title='theta_3d'):
    theta = all_theta_latent.cpu().detach() if hasattr(all_theta_latent, 'cpu') else all_theta_latent
    fig = plt.figure(figsize=(12, 4))
    
    # 3D plot
    ax1 = fig.add_subplot(131, projection='3d')
    scatter1 = ax1.scatter(theta[:, 0], theta[:, 1], theta[:, 2], c=all_alpha, cmap='viridis', s=20)
    ax1.set_xlabel('θ₀'), ax1.set_ylabel('θ₁'), ax1.set_zlabel('θ₂')
    
    # 2D projections
    ax2 = fig.add_subplot(132)
    scatter2 = ax2.scatter(theta[:, 0], theta[:, 1], c=all_alpha, cmap='viridis', s=20)
    ax2.set_xlabel('θ₀'), ax2.set_ylabel('θ₁')
    
    ax3 = fig.add_subplot(133)
    scatter3 = ax3.scatter(theta[:, 0], theta[:, 2], c=all_alpha, cmap='viridis', s=20)
    ax3.set_xlabel('θ₀'), ax3.set_ylabel('θ₂')
    
    plt.colorbar(scatter2, ax=[ax1, ax2, ax3], label='Alpha', shrink=0.8)
    
    plt.tight_layout()
    Path(f"plots/{run_name}").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"plots/{run_name}/{title}.png", dpi=300, bbox_inches='tight')

# Usage: plot_3d_scatter(all_theta_latent, all_alpha, run_name)

In [ ]:
#all_theta_latent = torch.cat(all_theta_latent)
all_theta_latent = lit_model.codebook
#all_alpha = [1]*1024 + [2]*1024 + [3]*1024


In [ ]:
plot_3d_scatter(all_theta_latent, all_alpha, run_name=run_name)

In [ ]:
plt.scatter(all_theta_latent[:, 0].cpu().detach(), all_theta_latent[:, 1].cpu().detach())

In [ ]:
plt.scatter(all_theta_latent[:, 1].cpu().detach(), all_theta_latent[:, 2].cpu().detach())

In [ ]:
with torch.no_grad():
    all_theta_latent_expanded = model.decoder_common(all_theta_latent)

In [ ]:
plt.plot(all_alpha, all_theta_latent_expanded[:, 4].cpu().detach())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Your theta encodings from the neural network (replace with actual data)
theta = all_theta_latent# Shape (n, 3) - replace with your actual theta


In [ ]:
# 1. OOD Dataset Loader
def load_ood_dataset(ood_type, split='train'):
  """Load specific OOD dataset"""
  file_path = f"/mnt/home/lserrano/disco-ball/datasets/combined_equation/ood/{ood_type}_{split}_512.h5"
  dataset = HDF5TemporalDataset([file_path], N_INPUT_FRAMES, N_OUTPUT_FRAMES, SUB_X, SUB_T, split)
  dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
  return dataloader

# 2. Simple Direct Prediction Test
def test_direct_prediction(model, dataloader, ood_name):
  """Test standard model prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          pred, _ = model(inp, state_labels, n_future_steps=target.shape[1])#, integration_time=target.shape[1])
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Direct Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 3. Simple Theta Prediction Test  
def test_theta_prediction(model, dataloader, ood_name):
  """Test theta-based prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          # Extract theta from encoder
          theta_latent, metadata = model.encode_theta_latent(inp, state_labels)
          theta = model.decode_theta(theta_latent, dim=1)

          # Predict using theta
          pred, _ = model.solve_ode(inp[:, -1], theta, state_labels, dim=1,
                                  n_future_steps=target.shape[1], )#integration_time=target.shape[1], dt=1,
                                   #predict_normed=False, metadata={})
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Theta Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 4. Simple Greedy Test (using first batch only for speed)
def test_greedy_composition(model, dataloader, encoded_operators, ood_name):
    """Test greedy search on first batch"""
    first_batch = next(iter(dataloader))
    
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    alpha = first_batch['alpha']
    beta = first_batch['beta']
    gamma = first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = greedy_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], max_operators=5)
        alphas = sum([metadata[op_id]["alpha"] for op_id in composition])
        betas = sum([metadata[op_id]["beta"] for op_id in composition])
        gammas = sum([metadata[op_id]["gamma"] for op_id in composition])
        print(f"{ood_name} Best Greedy Composition: {composition}")
        print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    return composition, torch.cat(pred_all)

# 5. Simple Random Test  
def test_random_composition(model, dataloader, encoded_operators, ood_name, num_compositions=500, composition_lengths=[1,2,3,4,5]):
    """Test random search on first batch"""
    first_batch = next(iter(dataloader))
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    alpha = first_batch['alpha']
    beta = first_batch['beta']
    gamma = first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = random_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], num_compositions=num_compositions, composition_lengths=composition_lengths)
        alphas = sum([metadata[op_id]["alpha"] for op_id in composition])
        betas = sum([metadata[op_id]["beta"] for op_id in composition])
        gammas = sum([metadata[op_id]["gamma"] for op_id in composition])
        print(f"{ood_name} Best Random Composition: {composition}")
        print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    
    return composition, torch.cat(pred_all), target
    
def test_nearest_param_selection(model, dataloader, encoded_operators, ood_name, num_integration_steps=1):
      """Test nearest parameter selection on first batch"""
      first_batch = next(iter(dataloader))

      inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
      alpha = first_batch['alpha']
      beta = first_batch['beta']
      gamma = first_batch['gamma']
      theta_operators, _, metadata = encoded_operators

      # Ensure theta_operators is on the right device
      theta_operators = theta_operators.to(device)

      state_labels = torch.tensor([0], device=device)
      loss_fn = RelativeL2()

      prediction = []

      for i in range(inp.shape[0]):
          print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")

          # Find nearest parameter composition
          composition = nearest_param_selection(alpha[i], beta[i], gamma[i], metadata)

          if composition:
              # Test the composition
              model.eval()
              with torch.no_grad():
                  try:
                      pred = strang_splitting_composition(
                          inp[i:i+1, -1], state_labels, composition, theta_operators, model,
                          integration_time=4/250, n_future_steps=target.shape[1], num_integration_steps=num_integration_steps
                      )
                      if pred.ndim==3:
                          pred = pred.unsqueeze(0)
                      pred = rearrange(pred, 't b c h -> b t c h')
                      prediction.append(pred)
                      test_error = loss_fn(pred, target[i:i+1]).item()

                      # Get the actual parameters of the selected operators
                      found_alphas = [metadata[op_idx]["alpha"] for op_idx in composition]
                      found_betas = [metadata[op_idx]["beta"] for op_idx in composition]
                      found_gammas = [metadata[op_idx]["gamma"] for op_idx in composition]

                      # Convert to numpy if needed
                      found_alphas = [x if hasattr(x, 'numpy') else x for x in found_alphas]
                      found_betas = [x if hasattr(x, 'numpy') else x for x in found_betas]
                      found_gammas = [x if hasattr(x, 'numpy') else x for x in found_gammas]

                      print(f"{ood_name} Nearest Param Composition: {composition}")
                      print(f"Parameters found: alpha={found_alphas}, beta={found_betas}, gamma={found_gammas}")
                      print(f"Test error: {test_error:.6f}")

                  except Exception as e:
                      print(f"Error applying composition {composition}: {e}")
          else:
              print(f"No suitable operator found (all target parameters are zero)")

          print(f"\n")

      return composition , torch.cat(prediction), target

# 6. Test All OOD Datasets
def test_all_ood_datasets(ood_types = ['E_BG']):
  """Test all methods on all OOD datasets"""
  #ood_types = ['E_ALL', 'E_BG', 'E_ED', 'E_HE']
  #ood_types = ['E_HE']
  
  results = {}
  

  for ood_type in ood_types:
      print(f"\n=== Testing {ood_type} ===")

  
      dataloader = load_ood_dataset(ood_type)
      target = []
      for batch in dataloader:
          target.append(batch['target'])
          

      # Test direct prediction
      direct_error, pred_direct = test_direct_prediction(model, dataloader, ood_type)

      # Test theta prediction  
      #theta_error, pred_theta = test_theta_prediction(model, dataloader, ood_type)

      # Test compositions (if operators available)
      if encoded_operators:
          #greedy_comp, pred_greedy = test_greedy_composition(model, dataloader, encoded_operators, ood_type)
          #random_comp, pred_random = test_random_composition(model, dataloader, encoded_operators, ood_type, composition_lengths=[2,3,4])
          nearest_comp, pred_nearest, target_nearest = test_nearest_param_selection(model, dataloader, encoded_operators, ood_type, num_integration_steps=5)
      else:
          greedy_comp = random_comp = None

      results[ood_type] = {
          #'direct_error': direct_error,
          #'theta_error': theta_error,
          #'greedy_composition': greedy_comp,
          #'random_composition': random_comp,
          #'pred_direct':pred_direct,
          #"pred_random":pred_random,
          #'pred_theta':pred_theta,
          #'pred_greedy':pred_greedy,
          'pred_nearest':pred_nearest,
          'target_nearest':target_nearest,
          'target': torch.cat(target),
      }

      #except Exception as e:
      #    print(f"Error testing {ood_type}: {e}")
      #    results[ood_type] = {'error': str(e)}

  return results

In [ ]:
# Operator Encoding

def encode_operators_from_training_data(model, train_files, num_operators=20, 
                                       n_trajectories_per_operator=4):
    """Encode operators from training trajectories."""
    if model is None:
        print("Model not loaded")
        return None
    
    print(f"Encoding {num_operators} operators from training data...")
    
    # Create training datasets
    train_datasets = {}
    for eq_type, file_path in train_files.items():
        if os.path.exists(file_path):
            dataset = HDF5TemporalDataset(
                hdf5_files=[file_path],
                input_frames=N_INPUT_FRAMES,
                output_frames=N_OUTPUT_FRAMES,
                sub_x=1, sub_t=1, split='train'
            )
            train_datasets[eq_type] = dataset
            print(f"  {eq_type}: {len(dataset)} training samples")
    
    if not train_datasets:
        print("No training datasets available")
        return None
    
    # Collect trajectories for encoding
    all_trajectories = []
    operator_metadata = []
    
    trajectories_per_equation = num_operators // len(train_datasets)
    remaining = num_operators % len(train_datasets)

    total_collected=0
    for eq_idx, (eq_type, dataset) in enumerate(train_datasets.items()):
        ops_from_this_eq = trajectories_per_equation + (1 if eq_idx < remaining else 0)
        
        print(f"Encoding {ops_from_this_eq} operators from {eq_type}...")
        
        dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
        
        collected = 0
        target_trajectories = ops_from_this_eq * n_trajectories_per_operator
        
        for batch in dataloader:
            if collected >= target_trajectories:
                break
                
            input_seq = batch['input']
            alpha = batch['alpha']
            beta = batch['beta']
            gamma = batch['gamma']
            
            for sample_idx in range(input_seq.shape[0]):
                if collected >= target_trajectories:
                    break
                
                trajectory = input_seq[sample_idx:sample_idx+1]
                all_trajectories.append(trajectory)
                
                # Track operator metadata
                operator_idx = collected #// n_trajectories_per_operator
                if collected % n_trajectories_per_operator == 0:
                    operator_metadata.append({
                        'operator_id': total_collected,
                        'equation_type': eq_type,
                        'trajectory_indices': [],
                        'alpha':alpha[sample_idx:sample_idx+1],
                        'beta':beta[sample_idx:sample_idx+1],
                        'gamma':gamma[sample_idx:sample_idx+1],
                    })
                
                operator_metadata[-1]['trajectory_indices'].append(len(all_trajectories) - 1)
                collected += 1
                total_collected +=1
    
    print(f"Collected {len(all_trajectories)} trajectories for {len(operator_metadata)} operators")
    
    # Encode all trajectories
    all_theta_latent = []
    all_theta = []
    
    state_labels = torch.tensor([0], device=device)
    encoding_batch_size = 32
    
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(all_trajectories), encoding_batch_size), desc="Encoding"):
            batch_trajectories = all_trajectories[i:i+encoding_batch_size]
            batch_input = torch.cat(batch_trajectories, dim=0).to(device)
            
            theta_latent_batch, _ = model.encode_theta_latent(batch_input, state_labels)
            theta_batch = model.decode_theta(theta_latent_batch, dim=1)
            
            all_theta_latent.append(theta_latent_batch.cpu())
            all_theta.append(theta_batch.cpu())
    
    all_theta_latent = torch.cat(all_theta_latent, dim=0)
    all_theta = torch.cat(all_theta, dim=0)
    
    # Average parameters for each operator
    num_unique_operators = len(operator_metadata)
    theta_operators = torch.zeros(num_unique_operators, all_theta.shape[1])
    theta_latent_operators = torch.zeros(num_unique_operators, all_theta_latent.shape[1])
    
    for op_idx, op_meta in enumerate(operator_metadata):
        traj_indices = op_meta['trajectory_indices']
        theta_operators[op_idx] = all_theta[traj_indices].mean(dim=0)
        theta_latent_operators[op_idx] = all_theta_latent[traj_indices].mean(dim=0)
    
    print(f"\nEncoded {num_unique_operators} operators:")
    print(f"  Theta shape: {theta_operators.shape}")
    print(f"  Theta latent shape: {theta_latent_operators.shape}")
    
    # Print distribution
    eq_counts = {}
    for op_meta in operator_metadata:
        eq_type = op_meta['equation_type']
        eq_counts[eq_type] = eq_counts.get(eq_type, 0) + 1
    
    print(f"  Distribution: {eq_counts}")
    
    return theta_operators, theta_latent_operators, operator_metadata

In [ ]:
def nearest_param_selection(target_alpha, target_beta, target_gamma, metadata):
  """
  For each non-zero target parameter, find the operator with the closest value.
  Combine all selected operators into a composition.
  
  Args:
      target_alpha: target alpha value (ignored if 0)
      target_beta: target beta value (ignored if 0)  
      target_gamma: target gamma value (ignored if 0)
      metadata: list/dict of operator metadata containing alpha, beta, gamma
      
  Returns:
      composition: list of operator indices (one for each non-zero target parameter)
  """
  import numpy as np

  # Convert targets to numpy if they're tensors
  if hasattr(target_alpha, 'numpy'):
      target_alpha = target_alpha.numpy()
  if hasattr(target_beta, 'numpy'):
      target_beta = target_beta.numpy()
  if hasattr(target_gamma, 'numpy'):
      target_gamma = target_gamma.numpy()

  composition = []

  # For each non-zero parameter, find the closest operator
  if target_alpha != 0:
      min_distance = float('inf')
      best_alpha_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_alpha = op_metadata['alpha']
          if hasattr(op_alpha, 'numpy'):
              op_alpha = op_alpha.numpy()

          distance = abs(target_alpha - op_alpha)
          if distance < min_distance:
              min_distance = distance
              best_alpha_op = op_idx

      if best_alpha_op is not None:
          composition.append(best_alpha_op)

  if target_beta != 0:
      min_distance = float('inf')
      best_beta_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_beta = op_metadata['beta']
          if hasattr(op_beta, 'numpy'):
              op_beta = op_beta.numpy()

          distance = abs(target_beta - op_beta)
          if distance < min_distance:
              min_distance = distance
              best_beta_op = op_idx

      if best_beta_op is not None:
          composition.append(best_beta_op)

  if target_gamma != 0:
      min_distance = float('inf')
      best_gamma_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_gamma = op_metadata['gamma']
          if hasattr(op_gamma, 'numpy'):
              op_gamma = op_gamma.numpy()

          distance = abs(target_gamma - op_gamma)
          if distance < min_distance:
              min_distance = distance
              best_gamma_op = op_idx

      if best_gamma_op is not None:
          composition.append(best_gamma_op)

  return composition

In [ ]:
def greedy_operator_selection(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running greedy operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    # Random validation timestep
    #val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    current_composition = []
    current_best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'greedy'
    }
    
    model.eval()
    
    for comp_length in range(1, max_operators + 1):
        print(f"\n--- Step {comp_length}: Testing all operators ---")
        
        best_error_for_step = float('inf')
        best_composition_for_step = None
        best_operator_added = None
        
        with torch.no_grad():
            for op_idx in range(num_operators):
                composition = current_composition + [op_idx]
                
                try:
                    #pred = sequential_operator_composition(
                    #    x_val, state_labels, composition, theta_operators, model,
                    #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                    #)
                     
                    pred = strang_splitting_composition(x_val, state_labels, composition, theta_operators, model,
                    integration_time=4/250, n_future_steps=1, num_integration_steps=1)#5)
                    
                    error = loss_fn(pred, y_val).item()
                    
                    history['compositions'].append(composition.copy())
                    history['errors'].append(error)
                    
                    if error < best_error_for_step:
                        best_error_for_step = error
                        best_composition_for_step = composition.copy()
                        best_operator_added = op_idx
                        
                except Exception as e:
                    continue
        
        if comp_length == 1:
            improvement = 0
            should_continue = True
        else:
            improvement = (current_best_error - best_error_for_step) / current_best_error * 100
            should_continue = improvement >= min_improvement_threshold
        
        print(f"  Best operator: {best_operator_added}, error: {best_error_for_step:.6f}")
        if comp_length > 1:
            print(f"  Improvement: {improvement:+.2f}%")
        
        if should_continue and best_error_for_step < current_best_error:
            current_composition = best_composition_for_step.copy()
            current_best_error = best_error_for_step
        else:
            print(f"  Stopping - insufficient improvement")
            break

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, current_composition, theta_operators, model,
                    integration_time=4/250, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=1
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Greedy selection completed: {current_composition} leading to error: {test_error:.6f}")
    #print(f"\nGreedy selection completed: {current_composition} (error: {current_best_error:.6f})")
    return current_composition, history, pred

In [ ]:
def random_operator_selection(model, theta_operators, test_input, test_target, 
                             num_compositions=1000, composition_lengths=[1, 2, 3, 4, 5]):
    """Random search for operator composition."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running random operator search...")
    print(f"Testing {num_compositions} random compositions")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    best_composition = []
    best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'random',
        'total_tested': 0
    }
    
    model.eval()
    
    with torch.no_grad():
        for comp_idx in tqdm(range(num_compositions), desc="Random search"):
            comp_length = random.choice(composition_lengths)
            composition = random.sample(range(num_operators), comp_length)
            
            try:
                #pred = sequential_operator_composition(
                #    x_val, state_labels, composition, theta_operators, model,
                #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                #)
                
                pred = strang_splitting_composition(
                    x_val, state_labels, composition, theta_operators, model,
                    integration_time=4/250, n_future_steps=1, num_integration_steps=1#5
                )
                
                error = loss_fn(pred, y_val).item()
                
                history['compositions'].append(composition.copy())
                history['errors'].append(error)
                history['total_tested'] += 1
                
                if error < best_error:
                    best_error = error
                    best_composition = composition.copy()
                    
            except Exception as e:
                continue

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, best_composition, theta_operators, model,
                    integration_time=4/250, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=1
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Random search completed: {best_composition} leading to error: {test_error:.6f}")
    print(f"Successfully tested: {history['total_tested']}/{num_compositions}")
    
    return best_composition, history, pred

In [ ]:
#train_files = [TRAINING_FILES[key] for key in TRAINING_FILES.keys()]
#theta_operators, theta_latent_operators, operator_metadata = encode_operators_from_training_data(model, TRAINING_FILES, num_operators=128*3, n_trajectories_per_operator=1)

In [ ]:
theta_latent_operators = lit_model.codebook
with torch.no_grad():
    theta_operators = model.decode_theta(theta_latent_operators, dim=1)

In [ ]:
# operator_metadata_full = []
cpt=0
operator_metadata = []
for key in ['EULER', 'HEAT', 'DISP']:
    dataset = train_datasets[key]
    for index in range(0, len(dataset), 64):
        sample = dataset.__getitem__(index)
        operator_metadata.append({
                        'operator_id': cpt,
                        'equation_type': key,
                        'trajectory_indices': [],
                        'alpha':sample['alpha'],
                        'beta':sample['beta'],
                        'gamma':sample['gamma']
                    })

In [ ]:
plt.ion()  # Turn on interactive mod

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Sample data
np.random.seed(42)
theta = np.random.randn(20, 3) * 0.05

fig = go.Figure(data=go.Scatter3d(
    x=theta[:, 0], y=theta[:, 1], z=theta[:, 2],
    mode='markers',
    marker=dict(size=8, color='blue')
))

fig.update_layout(title='Interactive 3D Plot')
fig.show(renderer="notebook")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots

# Your theta encodings from the neural network (replace with actual data)
theta = theta_latent_operators.detach().cpu()# Shape (n, 3) - replace with your actual theta

# Your metadata template with true parameters
template_info = operator_metadata

# Extract theta coordinates (neural network encodings)
theta_x = theta[:, 0]
theta_y = theta[:, 1] 
theta_z = theta[:, 2]

# Extract true parameters from metadata (handle PyTorch tensors)
true_alpha = [info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha'] for info in template_info]
true_beta = [info['beta'].item() if hasattr(info['beta'], 'item') else info['beta'] for info in template_info]
true_gamma = [info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma'] for info in template_info]

# Color mapping for equation types
color_map = {'EULER': '#FF6B6B', 'HEAT': '#4ECDC4', 'DISP': '#45B7D1', 'OTHER': '#96CEB4'}
colors = [color_map.get(info['equation_type'], color_map['OTHER']) for info in template_info]

# Get equation types
equation_types = [info['equation_type'] for info in template_info]

# Create hover text with parameter information
hover_texts = []
for i, info in enumerate(template_info):
    alpha_val = info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha']
    beta_val = info['beta'].item() if hasattr(info['beta'], 'item') else info['beta']
    gamma_val = info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma']
    
    hover_text = f"""Equation Type: {info['equation_type']}
θ₁: {theta_x[i]:.4f}
θ₂: {theta_y[i]:.4f}
θ₃: {theta_z[i]:.4f}
True α: {alpha_val:.4f}
True β: {beta_val:.4f}
True γ: {gamma_val:.4f}"""
    hover_texts.append(hover_text)

# Create the main 3D scatter plot
fig = go.Figure()

# Add traces for each equation type
unique_types = list(set(equation_types))
for eq_type in unique_types:
    # Get indices for this equation type
    indices = [i for i, t in enumerate(equation_types) if t == eq_type]
    
    if indices:
        fig.add_trace(go.Scatter3d(
            x=[theta_x[i] for i in indices],
            y=[theta_y[i] for i in indices],
            z=[theta_z[i] for i in indices],
            mode='markers',
            name=eq_type,
            text=[hover_texts[i] for i in indices],
            hovertemplate='%{text}<extra></extra>',
            marker=dict(
                size=8,
                color=color_map.get(eq_type, color_map['OTHER']),
                opacity=0.8,
                line=dict(width=2, color='black')
            )
        ))

# Update layout for better visualization
fig.update_layout(
    title=dict(
        text='Neural Network Encodings (θ) vs True Parameters (α,β,γ)',
        x=0.5,
        font=dict(size=16)
    ),
    scene=dict(
        xaxis_title='θ₁ (Neural Encoding Dim 1)',
        yaxis_title='θ₂ (Neural Encoding Dim 2)',
        zaxis_title='θ₃ (Neural Encoding Dim 3)',
        camera=dict(
            eye=dict(x=1.2, y=1.2, z=1.2)
        ),
        bgcolor='white',
        xaxis=dict(gridcolor='lightgray', showbackground=True, backgroundcolor='white'),
        yaxis=dict(gridcolor='lightgray', showbackground=True, backgroundcolor='white'),
        zaxis=dict(gridcolor='lightgray', showbackground=True, backgroundcolor='white')
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50),
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='gray',
        borderwidth=1
    )
)

# Add annotation with statistics
stats_text = f"""Encoding Statistics:
θ₁: [{theta_x.min():.3f}, {theta_x.max():.3f}]
θ₂: [{theta_y.min():.3f}, {theta_y.max():.3f}]
θ₃: [{theta_z.min():.3f}, {theta_z.max():.3f}]

True Parameter Ranges:
α: [{min(true_alpha):.3f}, {max(true_alpha):.3f}]
β: [{min(true_beta):.3f}, {max(true_beta):.3f}]
γ: [{min(true_gamma):.3f}, {max(true_gamma):.3f}]"""

fig.add_annotation(
    text=stats_text,
    align="left",
    showarrow=False,
    xref="paper", yref="paper",
    x=0.02, y=0.02,
    bordercolor="gray",
    borderwidth=1,
    bgcolor="rgba(255,255,255,0.9)",
    font=dict(size=10)
)

# Show the interactive plot
fig.show(renderer="notebook")

# Create correlation subplots
fig_corr = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Neural Encoding vs True Alpha', 
                   'Neural Encoding vs True Beta', 
                   'Neural Encoding vs True Gamma'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}]]
)

# Add correlation plots
for eq_type in unique_types:
    indices = [i for i, t in enumerate(equation_types) if t == eq_type]
    
    if indices:
        # Alpha correlation
        fig_corr.add_trace(
            go.Scatter(
                x=[true_alpha[i] for i in indices],
                y=[theta_x[i] for i in indices],
                mode='markers',
                name=eq_type,
                marker=dict(color=color_map.get(eq_type, color_map['OTHER']), size=8),
                showlegend=True if eq_type == unique_types[0] else False  # Only show legend once
            ),
            row=1, col=1
        )
        
        # Beta correlation
        fig_corr.add_trace(
            go.Scatter(
                x=[true_beta[i] for i in indices],
                y=[theta_y[i] for i in indices],
                mode='markers',
                name=eq_type,
                marker=dict(color=color_map.get(eq_type, color_map['OTHER']), size=8),
                showlegend=False
            ),
            row=1, col=2
        )
        
        # Gamma correlation
        fig_corr.add_trace(
            go.Scatter(
                x=[true_gamma[i] for i in indices],
                y=[theta_z[i] for i in indices],
                mode='markers',
                name=eq_type,
                marker=dict(color=color_map.get(eq_type, color_map['OTHER']), size=8),
                showlegend=False
            ),
            row=1, col=3
        )

# Update correlation plot layout
fig_corr.update_xaxes(title_text="True α", row=1, col=1)
fig_corr.update_yaxes(title_text="θ₁ (Encoding)", row=1, col=1)
fig_corr.update_xaxes(title_text="True β", row=1, col=2)
fig_corr.update_yaxes(title_text="θ₂ (Encoding)", row=1, col=2)
fig_corr.update_xaxes(title_text="True γ", row=1, col=3)
fig_corr.update_yaxes(title_text="θ₃ (Encoding)", row=1, col=3)

fig_corr.update_layout(
    height=400,
    width=1200,
    title_text="Parameter Correlations",
    showlegend=True
)

fig_corr.show(renderer="notebook")

print("Interactive Plotly plots created!")
print("Controls:")
print("- Mouse drag: Rotate 3D plot")
print("- Mouse wheel: Zoom")
print("- Double-click: Reset view")
print("- Hover over points for detailed information")
print("- Click legend items to hide/show traces")
print(f"Neural encoding shape: {theta.shape}")
print("This shows how your neural network has learned to encode the true physical parameters.")

In [ ]:
#operator_metadata_full[:32]

In [ ]:
#all_theta_latent = lit_model.codes

In [ ]:
#index = torch.randperm(all_theta_latent.shape[0])

#theta_latent_operators = all_theta_latent.clone()[index][:1024]
#with torch.no_grad():
#    theta_operators = model.decode_theta(theta_latent_operators, dim=1)

In [ ]:
#indices = index.tolist()[:1024]
#operator_metadata = [operator_metadata_full[i] for i in indices]
 

In [ ]:
#encode_operators = 

In [ ]:
encoded_operators = theta_operators, theta_latent_operators, operator_metadata 

In [ ]:
alphas = np.array([m['alpha'] for m in operator_metadata]).squeeze()
betas = np.array([m['beta'] for m in operator_metadata]).squeeze()
gammas = np.array([m['gamma'] for m in operator_metadata]).squeeze()
alphas.sort()
betas.sort()
gammas.sort()

In [ ]:
# Usage:
#N_OUTPUT_FRAMES=50
#results = test_all_ood_datasets(['E_ALL'])
#results = test_all_ood_datasets(['E_BG'])
#results = test_all_ood_datasets(['E_ED'])
results = test_all_ood_datasets(['E_ALL'])

In [ ]:
N_OUTPUT_FRAMES

In [ ]:
# HEAT + BURGERS does not work it seems

In [ ]:
# try do do nearest neighbor parameter by parameter and see if it works

In [ ]:
#results['E_BG']

In [ ]:
target = results['E_ED']['target_nearest']

In [ ]:
u_composition = results['E_ED']['pred_nearest']

In [ ]:
idx=1

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(u_composition[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

## Operator Encoding

In [ ]:
# 1. OOD Dataset Loader
def load_ood_dataset(ood_type, split='train'):
  """Load specific OOD dataset"""
  file_path = f"/mnt/home/lserrano/disco-ball/datasets/combined_equation/ood/{ood_type}_{split}_512.h5"
  dataset = HDF5TemporalDataset([file_path], N_INPUT_FRAMES, N_OUTPUT_FRAMES, SUB_X, SUB_T, split)
  dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
  return dataloader

## Operator Selection Methods

In [ ]:
def simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang"):
    
    state_labels = torch.tensor([0], device=x.device)
    #state_labels = 0
    dim = 1
    
    small_dt = dt/refinement_factor
    small_dt_half = small_dt/2

    theta_1 = model.decode_theta(theta_latent_1, dim)
    theta_2 = model.decode_theta(theta_latent_2, dim)

    pred = x
    trajectory_pred = []
    for t_idx in range(nt):
        for _ in range(refinement_factor):
            if splitting_method == 'strang':
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred = pred[:, -1]

            else:  # lie
                # Lie splitting: full step op1, full step op2
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred = pred[:, -1]

        # Store prediction at each time step
        trajectory_pred.append(pred)

    # Convert trajectory to numpy array: (n_t+1, batch_size, 2, n_x, n_y)
    trajectory_pred = torch.cat(trajectory_pred, axis=1)
    return trajectory_pred


In [ ]:
import torch
import numpy as np
from typing import List, Tuple
from torch import nn

class RelativeL2(nn.Module):
    def forward(self, x, y):
        x = rearrange(x, "b ... -> b (...)")
        y = rearrange(y, "b ... -> b (...)")
        diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
        y_norms = torch.linalg.norm(y, ord=2, dim=-1)
        return (diff_norms / y_norms)


class DiscreteOperatorEvolution:
    def __init__(self, codebook, population_size=50, elite_size=10, mutation_rate=0.2):
        self.codebook = codebook  # [n_codes, latent_dim]
        self.n_codes = len(codebook)
        self.population_size = population_size
        self.elite_size = elite_size
        self.mutation_rate = mutation_rate
        
        # Population: each individual is (op1_index, op2_index)
        self.population = self._initialize_population()
        self.fitness_history = []
        self.relative_l2_error = RelativeL2()
        
    def _initialize_population(self):
        """Initialize population of discrete operator index pairs"""
        population = []
        for _ in range(self.population_size):
            op1_idx = np.random.randint(0, self.n_codes)
            op2_idx = np.random.randint(0, self.n_codes)
            population.append((op1_idx, op2_idx))
        return population
    
    def _decode_individual(self, individual):
        """Convert indices to actual operator embeddings"""
        theta_1 = self.codebook[individual[:, 0]]
        theta_2 = self.codebook[individual[:, 1]]
        return theta_1, theta_2
    
    def evaluate_fitness(self, individual, model, x_val, y_val):
        """Evaluate fitness of an individual"""
        #theta_1, theta_2 = self._decode_individual(individual)
        theta_1, theta_2 = self._decode_individual(individual)
        
        # Expand for batch processing if needed
        #if x_val.dim() > 1:
            #theta_1 = theta_1.unsqueeze(0).expand(x_val.shape[0], -1)
            #theta_2 = theta_2.unsqueeze(0).expand(x_val.shape[0], -1)
        
        with torch.no_grad():
            x_val = x_val.repeat(theta_1.shape[0], 1, 1)
            pred = simple_splitting(model, theta_1, theta_2, x_val, nt=1, dt=4/250, 
                                  refinement_factor=5, splitting_method="strang")
            #print(pred)
            loss = self.relative_l2_error(pred, y_val).cpu().numpy()
        return -loss
    
    def crossover(self, parent1, parent2):
        """Single-point crossover for discrete indices"""
        op1_p1, op2_p1 = parent1
        op1_p2, op2_p2 = parent2
        
        # Simple crossover: swap one operator
        if np.random.rand() < 0.5:
            child1 = (op1_p1, op2_p2)
            child2 = (op1_p2, op2_p1)
        else:
            child1 = (op1_p2, op2_p1)
            child2 = (op1_p1, op2_p2)
        
        return child1, child2
    
    def mutate(self, individual):
        """Discrete mutation: randomly change operator indices"""
        op1_idx, op2_idx = individual
        
        # Mutate operator 1
        if np.random.rand() < self.mutation_rate:
            op1_idx = np.random.randint(0, self.n_codes)
        
        # Mutate operator 2
        if np.random.rand() < self.mutation_rate:
            op2_idx = np.random.randint(0, self.n_codes)
        
        return (op1_idx, op2_idx)
    
    def evolve_generation(self, model, x_val, y_val):
        """Evolve one generation"""
        # Evaluate fitness
        fitness_scores = []
        #for individual in self.population:
        pop = torch.tensor(self.population)
        fitness_scores = self.evaluate_fitness(pop, model, x_val, y_val)
        #fitness_scores.append(fitness)
        
        # Sort by fitness
        sorted_indices = np.argsort(fitness_scores)[::-1]  # descending
        
        # Select elite
        elite = [self.population[i] for i in sorted_indices[:self.elite_size]]
        
        # Generate new population
        new_population = elite.copy()
        
        while len(new_population) < self.population_size:
            # Tournament selection
            parent1 = self._tournament_select(fitness_scores)
            parent2 = self._tournament_select(fitness_scores)
            
            # Crossover
            child1, child2 = self.crossover(parent1, parent2)
            
            # Mutation
            child1 = self.mutate(child1)
            child2 = self.mutate(child2)
            
            new_population.extend([child1, child2])
        
        self.population = new_population[:self.population_size]
        self.fitness_history.append(max(fitness_scores))
        
        return max(fitness_scores), elite[0]  # best fitness and best individual
    
    def _tournament_select(self, fitness_scores, tournament_size=3):
        """Tournament selection"""
        indices = np.random.choice(len(fitness_scores), tournament_size, replace=False)
        winner_idx = indices[np.argmax([fitness_scores[i] for i in indices])]
        return self.population[winner_idx]


In [ ]:
N_OUTPUT_FRAMES=50
dataloader = load_ood_dataset("E_HE")

In [ ]:
for batch in dataloader:
    test_input = batch["input"]
    test_target = batch["target"]
    break

In [ ]:
idx = 0
codebook = lit_model.codebook.detach().clone()
discrete_evo = DiscreteOperatorEvolution(codebook, population_size=64, elite_size=8, mutation_rate=0.2)
#idx = random.randint(0, test_input.shape[0])
x = test_input[idx, -1].unsqueeze(0).cuda()
y = test_target[idx].unsqueeze(0).cuda()
epochs=200

for generation in range(epochs):
    t = random.randint(0, test_input.shape[1]-2)
    x_val = test_input[idx, t].cuda().unsqueeze(0)
    y_val = test_input[idx, t+1].cuda().unsqueeze(0)
    
    # Evolve population
    best_fitness, best_individual = discrete_evo.evolve_generation(model, x_val, y_val)
    
    # Get best operators for this generation
    theta_latent_1, theta_latent_2 = discrete_evo._decode_individual(torch.tensor(best_individual).unsqueeze(0))
    
    if generation % 20 == 0:
        print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
        print(f"Best operators: {best_individual}")
        #print('theta_latent_1', theta_latent_1.shape, theta_latent_1.dtype)

    if generation == epochs-1:
        with torch.no_grad():
            pred = simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
        error = discrete_evo.relative_l2_error(pred, y)
        print(f"Extrapolation error: {error.mean()}")

        

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred.squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(y.squeeze().cpu().detach()[t])

In [ ]:
codebook = lit_model.codebook.detach().clone()

all_theta_latent_1 = []
all_theta_latent_2 = []
for idx in range(32):
    discrete_evo = DiscreteOperatorEvolution(codebook, population_size=64, elite_size=8, mutation_rate=0.2)
    x = test_input[idx, -1].unsqueeze(0).cuda()
    y = test_target[idx].unsqueeze(0).cuda()
    epochs=100
    for generation in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[idx, t].cuda().unsqueeze(0)
        y_val = test_input[idx, t+1].cuda().unsqueeze(0)
        
        # Evolve population
        best_fitness, best_individual = discrete_evo.evolve_generation(model, x_val, y_val)
        
        # Get best operators for this generation
        theta_latent_1, theta_latent_2 = discrete_evo._decode_individual(torch.tensor(best_individual).unsqueeze(0))
        
        #if generation % 20 == 0:
            #pass
            #print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
            #print(f"Best operators: {best_individual}")
            #print('theta_latent_1', theta_latent_1.shape, theta_latent_1.dtype)
    
        if generation == epochs-1:
            print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
            print(f"Best operators: {best_individual}")
        #    with torch.no_grad():
        #        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
        #    error = discrete_evo.relative_l2_error(pred, y)
        #    print(f"Extrapolation error: {error.mean()}")
            
    all_theta_latent_1.append(theta_latent_1)
    all_theta_latent_2.append(theta_latent_2)
        
all_theta_latent_1 = torch.cat(all_theta_latent_1)
all_theta_latent_2 = torch.cat(all_theta_latent_2)

In [ ]:
with torch.no_grad():
    pred = simple_splitting(model, all_theta_latent_1, all_theta_latent_2, test_input[:,-1].cuda(), nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
    error = discrete_evo.relative_l2_error(pred, test_target.cuda())
    print(f"Extrapolation error: {error.mean()}")

In [ ]:
# ED: 0.05644378811120987
# BG: 0.003506791777908802
# HE: 0.030499424785375595

In [ ]:
def multi_operator_splitting(model, theta_latents, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang"):
    """
    Generalized operator splitting for any number of operators
    
    Args:
        model: The model with solve_ode and decode_theta methods
        theta_latents: List of theta latent vectors [theta_1, theta_2, ..., theta_k]
        x: Input tensor
        nt: Number of time steps
        dt: Time step size
        refinement_factor: Refinement factor for sub-stepping
        splitting_method: 'strang', 'lie', or 'symmetric'
    
    Returns:
        trajectory_pred: Predicted trajectory
    """
    
    state_labels = torch.tensor([0], device=x.device)
    dim = 1
    
    small_dt = dt / refinement_factor
    k_operators = len(theta_latents)
    
    # Decode all operators
    thetas = [model.decode_theta(theta_latent, dim) for theta_latent in theta_latents]
    
    pred = x
    trajectory_pred = []
    
    for t_idx in range(nt):
        for ref_step in range(refinement_factor):
            
            if splitting_method == 'strang':
                # Strang splitting for k operators: 
                # dt/2 * op_1, dt/2 * op_2, ..., dt/2 * op_{k-1}, dt * op_k, dt/2 * op_{k-1}, ..., dt/2 * op_2, dt/2 * op_1
                
                # Forward pass (first half steps for all but last operator)
                for i in range(k_operators - 1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
                
                # Full step for last operator
                pred, metadata = model.solve_ode(pred, thetas[-1], state_labels, dim, 
                                               n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred = pred[:, -1]
                
                # Backward pass (second half steps in reverse order)
                for i in range(k_operators - 2, -1, -1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
            
            elif splitting_method == 'lie':
                # Lie splitting: sequential full steps
                for theta in thetas:
                    pred, metadata = model.solve_ode(pred, theta, state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt, dt=small_dt)
                    pred = pred[:, -1]
            
            elif splitting_method == 'symmetric':
                # Symmetric splitting for even number of operators
                if k_operators % 2 != 0:
                    raise ValueError("Symmetric splitting requires even number of operators")
                
                # First half operators with half time step
                for i in range(k_operators // 2):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
                
                # Second half operators with full time step  
                for i in range(k_operators // 2, k_operators):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt, dt=small_dt)
                    pred = pred[:, -1]
                
                # First half operators again with half time step (reverse order)
                for i in range(k_operators // 2 - 1, -1, -1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
            
            else:
                raise ValueError(f"Unknown splitting method: {splitting_method}")
        
        # Store prediction at each time step
        trajectory_pred.append(pred)
    
    # Convert trajectory to tensor
    trajectory_pred = torch.cat(trajectory_pred, dim=1)
    return trajectory_pred

In [ ]:
class MultiOperatorEvolution:
    def __init__(self, codebook, n_operators=2, population_size=50, elite_size=10, mutation_rate=0.2):
        self.codebook = codebook  # [n_codes, latent_dim]
        self.n_codes = len(codebook)
        self.n_operators = n_operators  # Number of operators to combine
        self.population_size = population_size
        self.elite_size = elite_size
        self.mutation_rate = mutation_rate
        
        # Population: each individual is a list of operator indices [op1_idx, op2_idx, ..., opk_idx]
        self.population = self._initialize_population()
        self.fitness_history = []
        self.relative_l2_error = RelativeL2()
        
    def _initialize_population(self):
        """Initialize population of operator index combinations"""
        population = []
        for _ in range(self.population_size):
            individual = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
            population.append(individual)
        return population
    
    def _decode_individual_batch(self, population_batch):
        """Convert batch of individuals to operator embeddings"""
        # population_batch: [batch_size, n_operators]
        batch_size = population_batch.shape[0]
        
        # Get embeddings for all operators in all individuals
        operator_embeddings = []
        for op_idx in range(self.n_operators):
            # Get indices for this operator across all individuals
            indices = population_batch[:, op_idx]  # [batch_size]
            embeddings = self.codebook[indices]    # [batch_size, latent_dim]
            operator_embeddings.append(embeddings)
        
        return operator_embeddings  # List of [batch_size, latent_dim] tensors
    
    def evaluate_fitness_batch(self, population_batch, model, x_val, y_val):
        """Evaluate fitness for a batch of individuals"""
        operator_embeddings = self._decode_individual_batch(population_batch)
        batch_size = population_batch.shape[0]
        
        with torch.no_grad():
            # Expand input for batch processing
            x_val_batch = x_val.repeat(batch_size, 1, 1)
            
            # Run multi-operator splitting
            pred = multi_operator_splitting(model, operator_embeddings, x_val_batch, 
                                          nt=y_val.shape[1], dt=4/250, refinement_factor=5, 
                                          splitting_method="strang")
            
            # Compute fitness (negative loss)
            losses = self.relative_l2_error(pred, y_val).cpu().numpy()
            
        return -losses  # Return as fitness (higher is better)
    
    def crossover(self, parent1, parent2):
        """Multi-point crossover for operator sequences"""
        child1 = []
        child2 = []
        
        for i in range(self.n_operators):
            if np.random.rand() < 0.5:
                child1.append(parent1[i])
                child2.append(parent2[i])
            else:
                child1.append(parent2[i])
                child2.append(parent1[i])
        
        return child1, child2
    
    def mutate(self, individual):
        """Discrete mutation: randomly change operator indices"""
        mutated = individual.copy()
        
        for i in range(self.n_operators):
            if np.random.rand() < self.mutation_rate:
                mutated[i] = np.random.randint(0, self.n_codes)
        
        return mutated
    
    def evolve_generation(self, model, x_val, y_val):
        """Evolve one generation with batch evaluation"""
        # Convert population to tensor for batch processing
        pop_tensor = torch.tensor(self.population)  # [population_size, n_operators]
        
        # Evaluate fitness for entire population at once
        fitness_scores = self.evaluate_fitness_batch(pop_tensor, model, x_val, y_val)
        
        # Sort by fitness (descending order - higher is better)
        sorted_indices = np.argsort(fitness_scores)[::-1]
        
        # Select elite
        elite = [self.population[i] for i in sorted_indices[:self.elite_size]]
        
        # Generate new population
        new_population = elite.copy()
        
        while len(new_population) < self.population_size:
            # Tournament selection
            parent1 = self._tournament_select(fitness_scores)
            parent2 = self._tournament_select(fitness_scores)
            
            # Crossover
            child1, child2 = self.crossover(parent1, parent2)
            
            # Mutation
            child1 = self.mutate(child1)
            child2 = self.mutate(child2)
            
            new_population.extend([child1, child2])
        
        # Trim to exact population size
        self.population = new_population[:self.population_size]
        
        best_fitness = np.max(fitness_scores)
        best_individual = self.population[sorted_indices[0]]
        
        self.fitness_history.append(best_fitness)
        
        return best_fitness, best_individual
    
    def _tournament_select(self, fitness_scores, tournament_size=3):
        """Tournament selection"""
        indices = np.random.choice(len(fitness_scores), tournament_size, replace=False)
        winner_idx = indices[np.argmax([fitness_scores[i] for i in indices])]
        return self.population[winner_idx]
    
    def get_best_operators(self, individual):
        """Convert individual to actual operator embeddings"""
        return [self.codebook[idx] for idx in individual]

In [ ]:
# Usage example:
def run_multi_operator_evolution():
    """Example of how to use the multi-operator evolution"""
    
    # Initialize with 3 operators instead of 2
    codebook = model.quantizer.embedding.weight
    multi_evo = MultiOperatorEvolution(codebook, n_operators=3, population_size=30, 
                                     elite_size=6, mutation_rate=0.3)
    
    for generation in range(100):
        # Get random training example
        idx = np.random.randint(0, test_input.shape[0])
        t = np.random.randint(0, test_input.shape[1] - 2)
        x_val = test_input[idx, t].cuda().unsqueeze(0)
        y_val = test_input[idx, t+1].cuda().unsqueeze(0)
        
        # Evolve population
        best_fitness, best_individual = multi_evo.evolve_generation(model, x_val, y_val)
        
        if generation % 10 == 0:
            print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
            print(f"Best operators: {best_individual}")
    
    # Get final best operators
    final_best = multi_evo.population[0]  # Assuming sorted
    best_operators = multi_evo.get_best_operators(final_best)
    
    return best_operators, multi_evo

In [ ]:
N_OUTPUT_FRAMES=50
dataloader = load_ood_dataset("E_ALL")
for batch in dataloader:
    test_input = batch["input"]
    test_target = batch["target"]
    break

In [ ]:
codebook = lit_model.codebook.detach().clone()

all_best_operators = []

npreds = 4

for idx in range(64):
    print(idx)
    multi_evo = MultiOperatorEvolution(codebook, n_operators=3, population_size=64, 
                                     elite_size=16, mutation_rate=0.2)
    x = test_input[idx, -1].unsqueeze(0).cuda()
    y = test_target[idx].unsqueeze(0).cuda()
    epochs=100
    for generation in range(epochs):
        t = random.randint(0, test_input.shape[1]-npreds-1)
        #t = 14

        x_val = test_input[idx, t].cuda().unsqueeze(0)
        y_val = test_input[idx, t+1:t+npreds+1].cuda().unsqueeze(0)
        
        # Evolve population
        best_fitness, best_individual = multi_evo.evolve_generation(model, x_val, y_val)
        
        # Get best operators for this generation
        #theta_latent_1, theta_latent_2 = discrete_evo._decode_individual(torch.tensor(best_individual).unsqueeze(0))
        
        if generation % 20 == 0:
            #pass
            print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
            #print(f"Best operators: {best_individual}")
            #print('theta_latent_1', theta_latent_1.shape, theta_latent_1.dtype)
    
        if generation == epochs-1:
            print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
            print(f"Best operators: {best_individual}")
            
    best_operators = multi_evo.get_best_operators(best_individual)
    all_best_operators.append(best_operators)
    #print(len(best_operators, best_operators[0].shape))

In [ ]:
#all_best_operators = [best_individual]
len(all_best_operators)

In [ ]:
b=len(all_best_operators)
operators = rearrange(torch.stack([torch.stack(all_best_operators[i], axis=0) for i in range(b)], axis=0), 'b n c -> n b c')

In [ ]:
#all_best_operators[0][0]

In [ ]:
with torch.no_grad():
    #pred = simple_splitting(model, all_theta_latent_1, all_theta_latent_2, test_input[:,-1].cuda(), nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
    pred = multi_operator_splitting(model, operators, test_input[:7,-1].cuda(), nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
    error = multi_evo.relative_l2_error(pred, test_target[:7].cuda())
    print(f"Extrapolation error: {error.mean()}")

In [ ]:
error

In [ ]:
# only the last timestamp
#0.0147, 0.0082, 0.2001, 0.6857, 0.2586, 0.0553, 0.1280, 0.2930, 0.4670,
#        0.0484, 0.0095, 0.0066, 0.1324, 0.2298, 0.0393, 0.0132, 0.0856, 0.5372,
#        0.0046, 0.0222, 0.1212, 0.0766, 0.2061, 0.0485, 0.0056, 0.1484, 0.0132,
#        0.0019, 0.0100, 0.0039, 0.0405, 0.0904

#0.0674

#[0.0113, 0.0086, 0.0885, 0.4258, 0.0029, 0.0575, 0.1237, 0.0149, 0.0180,
#        0.0507, 0.0128, 0.1474, 0.0028, 0.0347, 0.0403, 0.0132, 0.0231, 0.1364,
#        0.0055, 0.0084, 0.2024, 0.2134, 0.0611, 0.0416, 0.0196, 0.1050, 0.0088,
#        0.0193, 0.0130, 0.0142, 0.1018, 0.1296]

In [ ]:
#0.0108, 0.0394, 0.0106, 0.0167
#[0.0159, 0.2276, 0.2518, 0.2631]
#

In [ ]:
idx=6
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(test_target[idx].squeeze().cpu().detach()[t])

In [ ]:
def create_simulator_for_your_interface(multi_operator_splitting):
    """
    Wrapper for your exact interface:
    multi_operator_splitting(model, theta_latents, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang")
    """
    def simulator_wrapper(model, operator_embeddings, x_val_batch, **kwargs):
        """
        Convert from optimizer interface to your interface
        
        Optimizer provides:
        - model: Your neural network
        - operator_embeddings: List of [batch_size, latent_dim] tensors
        - x_val_batch: [batch_size, ...] input tensor
        - **kwargs: nt, dt, refinement_factor, splitting_method
        
        Your function expects:
        - model: Same
        - theta_latents: List of [batch_size, latent_dim] tensors (same as operator_embeddings!)
        - x: [batch_size, ...] input tensor (same as x_val_batch!)
        - nt, dt, refinement_factor, splitting_method: Same
        """
        
        # Your function parameters - just rename the variables!
        theta_latents = operator_embeddings  # Same thing, different name
        x = x_val_batch                      # Same thing, different name
        
        # Extract kwargs with your defaults
        nt = kwargs.get('nt', 1)
        dt = kwargs.get('dt', 4/250) 
        refinement_factor = kwargs.get('refinement_factor', 5)
        splitting_method = kwargs.get('splitting_method', 'strang')
        
        # Call your function with exact interface
        return multi_operator_splitting(
            model=model,
            theta_latents=theta_latents,  # List of operator embeddings
            x=x,                          # Input batch
            nt=nt,
            dt=dt,
            refinement_factor=refinement_factor,
            splitting_method=splitting_method
        )
    
    return simulator_wrapper

# Create the wrapper
simulator = create_simulator_for_your_interface(multi_operator_splitting)

In [ ]:
import torch
import numpy as np
from typing import List, Tuple, Callable, Dict, Optional
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize, differential_evolution
import matplotlib.pyplot as plt
import time
import itertools

class DiscreteOperatorOptimizer:
    """Discrete sampling + GP regression for operator combination optimization"""
    
    def __init__(self, 
                 codebook: torch.Tensor,  # [n_codes, latent_dim]
                 simulator: Callable,     # Your multi-operator splitting function
                 n_operators: int = 2):
        
        self.codebook = codebook.detach().cpu().numpy()  # Convert to numpy
        self.n_codes, self.latent_dim = self.codebook.shape
        self.n_operators = n_operators
        self.simulator = simulator
        
        # GP regressor and scaler
        self.gp_regressor = None
        self.scaler = StandardScaler()
        
        # Data storage
        self.samples = []      # List of sampled operator combinations
        self.losses = []       # Corresponding losses
        self.sample_count = 0
        self.relative_l2_error = RelativeL2()
        
        print(f"Initialized optimizer with {self.n_codes} discrete operators, "
              f"{self.latent_dim}D latent space")
        print(f"Total possible combinations: {self.n_codes ** self.n_operators}")
        
    def sample_operator_combinations(self, 
                                   n_samples: int,
                                   sampling_strategy: str = 'random') -> List[List[int]]:
        """Sample operator index combinations from the discrete codebook"""
        
        total_combinations = self.n_codes ** self.n_operators
        
        if n_samples >= total_combinations:
            print(f"Requested {n_samples} samples >= {total_combinations} total combinations")
            print("Using exhaustive sampling instead")
            sampling_strategy = 'exhaustive'
        
        print(f"Generating {min(n_samples, total_combinations)} discrete samples using {sampling_strategy}...")
        
        if sampling_strategy == 'exhaustive':
            # Generate ALL possible combinations
            all_combinations = list(itertools.product(range(self.n_codes), repeat=self.n_operators))
            samples = [list(combo) for combo in all_combinations]
            print(f"Generated all {len(samples)} possible combinations")
            
        elif sampling_strategy == 'random':
            # Pure random sampling with replacement
            samples = []
            for _ in range(n_samples):
                combination = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
                samples.append(combination)
                
        elif sampling_strategy == 'random_without_replacement':
            # Random sampling without replacement
            if n_samples > total_combinations:
                n_samples = total_combinations
                print(f"Reducing samples to {n_samples} (maximum possible)")
            
            # Generate all combinations and randomly select subset
            all_combinations = list(itertools.product(range(self.n_codes), repeat=self.n_operators))
            selected_indices = np.random.choice(len(all_combinations), n_samples, replace=False)
            samples = [list(all_combinations[i]) for i in selected_indices]
            
        elif sampling_strategy == 'stratified':
            # Stratified sampling: ensure each operator appears roughly equally
            samples = []
            
            # For 2 operators, try to balance both dimensions
            if self.n_operators == 2:
                samples_per_op1 = n_samples // self.n_codes
                remaining_samples = n_samples % self.n_codes
                
                for op1_idx in range(self.n_codes):
                    n_for_this_op1 = samples_per_op1 + (1 if op1_idx < remaining_samples else 0)
                    
                    for _ in range(n_for_this_op1):
                        op2_idx = np.random.randint(0, self.n_codes)
                        samples.append([op1_idx, op2_idx])
                        
            else:
                # For more operators, fall back to random
                print("Stratified sampling only implemented for 2 operators, using random")
                return self.sample_operator_combinations(n_samples, 'random')
                
        elif sampling_strategy == 'diverse':
            # Try to maximize diversity in latent space
            samples = []
            
            # Start with a random sample
            first_combo = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
            samples.append(first_combo)
            
            # Greedily add samples that are far from existing ones
            for _ in range(n_samples - 1):
                best_combo = None
                best_min_distance = -1
                
                # Try multiple random candidates
                for _ in range(min(100, total_combinations)):
                    candidate = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
                    
                    # Compute minimum distance to existing samples
                    min_distance = float('inf')
                    for existing in samples:
                        # Distance in latent space
                        dist = 0
                        for op_idx in range(self.n_operators):
                            embed1 = self.codebook[candidate[op_idx]]
                            embed2 = self.codebook[existing[op_idx]]
                            dist += np.sum((embed1 - embed2) ** 2)
                        
                        min_distance = min(min_distance, dist)
                    
                    if min_distance > best_min_distance:
                        best_min_distance = min_distance
                        best_combo = candidate
                
                if best_combo is not None:
                    samples.append(best_combo)
                else:
                    # Fallback to random
                    samples.append([np.random.randint(0, self.n_codes) for _ in range(self.n_operators)])
        
        else:
            raise ValueError(f"Unknown sampling strategy: {sampling_strategy}")
        
        # Remove duplicates while preserving order
        seen = set()
        unique_samples = []
        for sample in samples:
            sample_tuple = tuple(sample)
            if sample_tuple not in seen:
                seen.add(sample_tuple)
                unique_samples.append(sample)
        
        print(f"Generated {len(unique_samples)} unique operator combinations")
        
        # Show some examples
        print("Example combinations:")
        for i, combo in enumerate(unique_samples[:5]):
            print(f"  {i+1}: Operators {combo}")
            
        return unique_samples
    
    def evaluate_combinations(self, 
                            combinations: List[List[int]],
                            x_val: torch.Tensor,
                            y_val: torch.Tensor,
                            model: torch.nn.Module,
                            batch_size: int = 200) -> List[float]:
        """Evaluate operator combinations using your simulator"""
        
        print(f"Evaluating {len(combinations)} operator combinations...")
        all_losses = []
        
        # Process in batches for efficiency
        for batch_start in range(0, len(combinations), batch_size):
            batch_end = min(batch_start + batch_size, len(combinations))
            batch_combinations = combinations[batch_start:batch_end]
            
            if batch_start % (batch_size * 5) == 0:
                print(f"  Processing batch {batch_start//batch_size + 1}/{(len(combinations)-1)//batch_size + 1}")
            
            try:
                # Convert combinations to operator embeddings
                batch_losses = self._evaluate_combination_batch(
                    batch_combinations, x_val, y_val, model)
                all_losses.extend(batch_losses)
                
            except Exception as e:
                print(f"Batch evaluation failed: {e}")
                # Add high loss for failed evaluations
                all_losses.extend([float('inf')] * len(batch_combinations))
        
        # Filter out infinite losses
        #print('losses', all_losses)
        valid_losses = [loss for loss in all_losses if np.isfinite(loss)]
        print(f"Valid evaluations: {len(valid_losses)}/{len(all_losses)}")
        if valid_losses:
            print(f"Loss range: {min(valid_losses):.6f} to {max(valid_losses):.6f}")
            
            # Show best combinations found
            loss_combo_pairs = [(loss, combinations[i]) for i, loss in enumerate(all_losses) if np.isfinite(loss)]
            loss_combo_pairs.sort(key=lambda x: x[0])
            
            print("Top 5 combinations found during evaluation:")
            for i, (loss, combo) in enumerate(loss_combo_pairs[:5]):
                print(f"  {i+1}: Loss {loss:.6f}, Operators {combo}")
        
        return all_losses
    
    def _evaluate_combination_batch(self, 
                                  combinations: List[List[int]], 
                                  x_val: torch.Tensor, 
                                  y_val: torch.Tensor,
                                  model: torch.nn.Module) -> List[float]:
        """Evaluate a batch of combinations using actual codebook operators"""
        
        batch_size = len(combinations)
        
        # Convert combinations to operator embeddings from codebook
        operator_embeddings = []
        for op_idx in range(self.n_operators):
            # Get indices for this operator across all combinations
            indices = [combo[op_idx] for combo in combinations]
            # Get actual embeddings from codebook
            embeddings = torch.tensor(self.codebook[indices], dtype=torch.float32, device=x_val.device)  # [batch_size, latent_dim]
            operator_embeddings.append(embeddings)
        
        # Expand input for batch processing
        x_val_batch = x_val.repeat(batch_size, 1, 1)

        #print(x_val_batch.shape, operator_embeddings[0].shape)

        #print(x_val_batch.device, operator_embeddings.device)
        # Run your multi-operator splitting simulator
        with torch.no_grad():
            pred = self.simulator(model, operator_embeddings, x_val_batch, 
                                nt=y_val.shape[1], dt=4/250, refinement_factor=5, 
                                splitting_method="strang")
            #print('pred', pred.shape)
            
            # Compute losses
            #losses = torch.mean((pred - y_val)**2, dim=(1, 2)).cpu().numpy()
            losses = self.relative_l2_error(pred, y_val).cpu().numpy()
        
        return losses.tolist()
    
    def fit_gp_regressor(self, 
                        combinations: List[List[int]], 
                        losses: List[float],
                        use_latent_features: bool = True) -> Dict:
        """Fit Gaussian Process regressor to the sampled data"""
        
        print("Fitting Gaussian Process regressor...")

        #print('losses', losses)
        
        # Filter valid samples
        valid_indices = [i for i, loss in enumerate(losses) if np.isfinite(loss)]
        
        if len(valid_indices) < 10:
            raise ValueError(f"Not enough valid samples ({len(valid_indices)}) to fit GP")
        
        valid_combinations = [combinations[i] for i in valid_indices]
        valid_losses = [losses[i] for i in valid_indices]
        
        if use_latent_features:
            # Use actual latent embeddings as features (much better than indices!)
            print("Using latent space embeddings as GP features")
            X = []
            for combo in valid_combinations:
                # Concatenate all operator embeddings for this combination
                combined_embedding = []
                for op_idx in combo:
                    combined_embedding.extend(self.codebook[op_idx])
                X.append(combined_embedding)
            X = np.array(X)
        else:
            # Use operator indices as features (less informative)
            print("Using operator indices as GP features")
            X = np.array(valid_combinations, dtype=float)
        
        y = np.array(valid_losses)
        
        # Standardize features
        X_scaled = self.scaler.fit_transform(X)
        
        # Set up kernel - use different length scales for latent vs index features
        if use_latent_features:
            # For latent features, use smaller length scale
            length_scale = 1.0
            kernel = (ConstantKernel(1.0, (1e-3, 1e3)) * 
                     RBF(length_scale, (1e-2, 1e2)))
        else:
            # For indices, use larger length scale
            length_scale = 2.0  
            kernel = (ConstantKernel(1.0, (1e-3, 1e3)) * 
                     RBF(length_scale, (1e-1, 10.0)))
        
        # Fit GP
        self.gp_regressor = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-6,
            normalize_y=True,
            n_restarts_optimizer=10,
            random_state=42
        )
        
        self.gp_regressor.fit(X_scaled, y)
        
        # Store data for future use
        self.samples = combinations
        self.losses = losses
        self.sample_count = len(valid_indices)
        self.use_latent_features = use_latent_features
        
        # Evaluate fit quality
        y_pred, y_std = self.gp_regressor.predict(X_scaled, return_std=True)
        
        from sklearn.metrics import r2_score, mean_squared_error
        r2 = r2_score(y, y_pred)
        mse = mean_squared_error(y, y_pred)
        
        results = {
            'r2_score': r2,
            'mse': mse,
            'n_samples': len(valid_indices),
            'mean_uncertainty': np.mean(y_std),
            'feature_type': 'latent_embeddings' if use_latent_features else 'operator_indices',
            'feature_dim': X.shape[1],
            'kernel_params': self.gp_regressor.kernel_.get_params()
        }
        
        print(f"GP Regressor fitted:")
        print(f"  R² score: {r2:.4f}")
        print(f"  MSE: {mse:.6f}")
        print(f"  Samples used: {len(valid_indices)}")
        print(f"  Feature type: {results['feature_type']}")
        print(f"  Feature dimension: {X.shape[1]}")
        print(f"  Mean uncertainty: {np.mean(y_std):.6f}")
        
        return results
    
    def find_optimal_combination(self, 
                                method: str = 'exhaustive_discrete',
                                n_candidates: int = 50000) -> Tuple[List[int], float, Dict]:
        """Find optimal operator combination using the fitted GP"""
        
        if self.gp_regressor is None:
            raise ValueError("Must fit GP regressor first using fit_gp_regressor()")
        
        print(f"Finding optimal combination using {method}...")
        
        if method == 'exhaustive_discrete':
            return self._exhaustive_discrete_search()
            
        elif method == 'random_discrete':
            return self._random_discrete_search(n_candidates)
            
        elif method == 'greedy_search':
            return self._greedy_discrete_search()
            
        else:
            raise ValueError(f"Unknown optimization method: {method}")
    
    def _exhaustive_discrete_search(self) -> Tuple[List[int], float, Dict]:
        """Evaluate GP on ALL possible operator combinations"""
        
        total_combinations = self.n_codes ** self.n_operators
        
        if total_combinations > 50000:
            print(f"Warning: {total_combinations} combinations is very large, this might be slow")
            print("Consider using 'random_discrete' method instead")
        
        # Generate all possible combinations
        all_combinations = list(itertools.product(range(self.n_codes), repeat=self.n_operators))
        
        # Convert to features
        if self.use_latent_features:
            X_all = []
            for combo in all_combinations:
                combined_embedding = []
                for op_idx in combo:
                    combined_embedding.extend(self.codebook[op_idx])
                X_all.append(combined_embedding)
            X_all = np.array(X_all)
        else:
            X_all = np.array(all_combinations, dtype=float)
        
        # Predict on all combinations
        X_all_scaled = self.scaler.transform(X_all)
        pred_means, pred_stds = self.gp_regressor.predict(X_all_scaled, return_std=True)
        
        # Find best
        best_idx = np.argmin(pred_means)
        best_combination = list(all_combinations[best_idx])
        best_pred_loss = pred_means[best_idx]
        
        # Get top 10 for analysis
        sorted_indices = np.argsort(pred_means)
        top_combinations = []
        for i in range(min(10, len(sorted_indices))):
            idx = sorted_indices[i]
            top_combinations.append({
                'combination': list(all_combinations[idx]),
                'predicted_loss': pred_means[idx],
                'uncertainty': pred_stds[idx]
            })
        
        info = {
            'predicted_loss': best_pred_loss,
            'uncertainty': pred_stds[best_idx],
            'n_combinations_evaluated': len(all_combinations),
            'top_10_combinations': top_combinations,
            'method': 'exhaustive_discrete'
        }
        
        print(f"Evaluated all {len(all_combinations)} combinations")
        print(f"Best combination: {best_combination} with predicted loss: {best_pred_loss:.6f}")
        
        return best_combination, best_pred_loss, info
    
    def _random_discrete_search(self, n_candidates: int) -> Tuple[List[int], float, Dict]:
        """Random search over discrete combinations"""
        
        # Generate random candidates
        candidates = self.sample_operator_combinations(n_candidates, 'random')
        
        # Convert to features
        if self.use_latent_features:
            X_candidates = []
            for combo in candidates:
                combined_embedding = []
                for op_idx in combo:
                    combined_embedding.extend(self.codebook[op_idx])
                X_candidates.append(combined_embedding)
            X_candidates = np.array(X_candidates)
        else:
            X_candidates = np.array(candidates, dtype=float)
        
        # Predict
        X_candidates_scaled = self.scaler.transform(X_candidates)
        pred_means, pred_stds = self.gp_regressor.predict(X_candidates_scaled, return_std=True)
        
        # Find best
        best_idx = np.argmin(pred_means)
        best_combination = candidates[best_idx]
        best_pred_loss = pred_means[best_idx]
        
        info = {
            'predicted_loss': best_pred_loss,
            'uncertainty': pred_stds[best_idx],
            'n_candidates_evaluated': len(candidates),
            'method': 'random_discrete'
        }
        
        return best_combination, best_pred_loss, info
    
    def _greedy_discrete_search(self) -> Tuple[List[int], float, Dict]:
        """Greedy search: optimize one operator at a time"""
        
        # Start with random combination
        current_combo = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
        
        improved = True
        iterations = 0
        
        while improved and iterations < 10:
            improved = False
            iterations += 1
            
            # Try improving each operator position
            for pos in range(self.n_operators):
                best_loss = float('inf')
                best_op = current_combo[pos]
                
                # Try all operators at this position
                for op_idx in range(self.n_codes):
                    test_combo = current_combo.copy()
                    test_combo[pos] = op_idx
                    
                    # Convert to features and predict
                    if self.use_latent_features:
                        combined_embedding = []
                        for idx in test_combo:
                            combined_embedding.extend(self.codebook[idx])
                        X_test = np.array([combined_embedding])
                    else:
                        X_test = np.array([test_combo], dtype=float)
                    
                    X_test_scaled = self.scaler.transform(X_test)
                    pred_loss = self.gp_regressor.predict(X_test_scaled)[0]
                    
                    if pred_loss < best_loss:
                        best_loss = pred_loss
                        best_op = op_idx
                
                # Update if improved
                if best_op != current_combo[pos]:
                    current_combo[pos] = best_op
                    improved = True
                    print(f"  Iteration {iterations}, position {pos}: improved to {best_op}")
        
        # Final prediction
        if self.use_latent_features:
            combined_embedding = []
            for idx in current_combo:
                combined_embedding.extend(self.codebook[idx])
            X_final = np.array([combined_embedding])
        else:
            X_final = np.array([current_combo], dtype=float)
            
        X_final_scaled = self.scaler.transform(X_final)
        final_pred, final_std = self.gp_regressor.predict(X_final_scaled, return_std=True)
        
        info = {
            'predicted_loss': final_pred[0],
            'uncertainty': final_std[0],
            'n_iterations': iterations,
            'method': 'greedy_discrete'
        }
        
        return current_combo, final_pred[0], info
    
    def visualize_results(self, figsize=(15, 10)):
        """Create visualizations of the sampling and optimization results"""
        
        if not self.samples or not self.losses:
            print("No data to visualize")
            return None
        
        fig, axes = plt.subplots(2, 3, figsize=figsize)
        
        # 1. Loss distribution
        valid_losses = [loss for loss in self.losses if np.isfinite(loss)]
        axes[0, 0].hist(np.log10(valid_losses), bins=30, alpha=0.7, edgecolor='black')
        axes[0, 0].set_xlabel('log₁₀(Loss)')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].set_title('Loss Distribution')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Operator usage frequency
        if self.n_operators <= 3:
            valid_samples = [self.samples[i] for i, loss in enumerate(self.losses) if np.isfinite(loss)]
            if valid_samples and self.n_operators == 2:
                sample_array = np.array(valid_samples)
                hist, xedges, yedges = np.histogram2d(sample_array[:, 0], sample_array[:, 1], 
                                                     bins=min(20, self.n_codes))
                im = axes[0, 1].imshow(hist.T, origin='lower', cmap='viridis', 
                                      extent=[0, self.n_codes, 0, self.n_codes])
                axes[0, 1].set_xlabel('Operator 1 Index')
                axes[0, 1].set_ylabel('Operator 2 Index')
                axes[0, 1].set_title('Operator Pair Frequency')
                plt.colorbar(im, ax=axes[0, 1])
            else:
                # Show individual operator frequencies
                all_ops = []
                for sample in valid_samples:
                    all_ops.extend(sample)
                
                op_counts = np.bincount(all_ops, minlength=self.n_codes)
                axes[0, 1].bar(range(self.n_codes), op_counts)
                axes[0, 1].set_xlabel('Operator Index')
                axes[0, 1].set_ylabel('Usage Count')
                axes[0, 1].set_title('Operator Usage Frequency')
        
        # 3. GP prediction quality (if available)
        if self.gp_regressor is not None:
            valid_indices = [i for i, loss in enumerate(self.losses) if np.isfinite(loss)]
            if valid_indices:
                valid_combinations = [self.samples[i] for i in valid_indices]
                y_valid = np.array([self.losses[i] for i in valid_indices])
                
                # Convert to features
                if self.use_latent_features:
                    X_valid = []
                    for combo in valid_combinations:
                        combined_embedding = []
                        for op_idx in combo:
                            combined_embedding.extend(self.codebook[op_idx])
                        X_valid.append(combined_embedding)
                    X_valid = np.array(X_valid)
                else:
                    X_valid = np.array(valid_combinations, dtype=float)
                
                X_scaled = self.scaler.transform(X_valid)
                y_pred, y_std = self.gp_regressor.predict(X_scaled, return_std=True)
                
                axes[0, 2].scatter(y_valid, y_pred, alpha=0.6, s=20)
                min_val = min(y_valid.min(), y_pred.min())
                max_val = max(y_valid.max(), y_pred.max())
                axes[0, 2].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
                axes[0, 2].set_xlabel('True Loss')
                axes[0, 2].set_ylabel('Predicted Loss')
                axes[0, 2].set_title('GP Prediction Quality')
                
                from sklearn.metrics import r2_score
                r2 = r2_score(y_valid, y_pred)
                axes[0, 2].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[0, 2].transAxes,
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # 4. Best combinations over samples
        valid_indices = [i for i, loss in enumerate(self.losses) if np.isfinite(loss)]
        if valid_indices:
            valid_losses_array = np.array([self.losses[i] for i in valid_indices])
            cumulative_best = np.minimum.accumulate(valid_losses_array)
            axes[1, 0].plot(range(1, len(cumulative_best)+1), cumulative_best)
            axes[1, 0].set_xlabel('Sample Number')
            axes[1, 0].set_ylabel('Best Loss So Far')
            axes[1, 0].set_title('Convergence Plot')
            axes[1, 0].grid(True, alpha=0.3)
            axes[1, 0].set_yscale('log')
        
        # 5. Loss vs operator diversity
        if len(valid_indices) > 0:
            diversities = []
            valid_losses_for_diversity = []
            
            for i in valid_indices:
                combo = self.samples[i]
                loss = self.losses[i]
                
                # Compute diversity as variance of operator embeddings
                embeddings = [self.codebook[op_idx] for op_idx in combo]
                if len(embeddings) > 1:
                    embeddings_array = np.array(embeddings)
                    diversity = np.mean(np.var(embeddings_array, axis=0))
                else:
                    diversity = 0.0
                    
                diversities.append(diversity)
                valid_losses_for_diversity.append(loss)
            
            axes[1, 1].scatter(diversities, valid_losses_for_diversity, alpha=0.6, s=20)
            axes[1, 1].set_xlabel('Operator Diversity')
            axes[1, 1].set_ylabel('Loss')
            axes[1, 1].set_title('Loss vs Operator Diversity')
            axes[1, 1].grid(True, alpha=0.3)
        
        # 6. Statistics summary
        if valid_indices:
            stats_text = f"""
            Total Samples: {len(self.samples)}
            Valid Samples: {len(valid_indices)}
            Success Rate: {len(valid_indices)/len(self.samples)*100:.1f}%
            
            Best Loss: {min(valid_losses):.2e}
            Mean Loss: {np.mean(valid_losses):.2e}
            Std Loss: {np.std(valid_losses):.2e}
            
            Operators: {self.n_codes}
            Combinations: {self.n_codes**self.n_operators}
            Coverage: {len(valid_indices)/(self.n_codes**self.n_operators)*100:.1f}%
            """
            
            if self.gp_regressor is not None:
                stats_text += f"\nGP R²: {r2:.3f}"
            
            axes[1, 2].text(0.1, 0.5, stats_text, transform=axes[1, 2].transAxes,
                           fontsize=9, verticalalignment='center',
                           bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
            axes[1, 2].set_xlim(0, 1)
            axes[1, 2].set_ylim(0, 1)
            axes[1, 2].axis('off')
            axes[1, 2].set_title('Statistics')
        
        plt.tight_layout()
        return fig

# Main workflow function
def run_discrete_operator_optimization(codebook: torch.Tensor,
                                     simulator: Callable,
                                     x_val: torch.Tensor,
                                     y_val: torch.Tensor,
                                     model: torch.nn.Module,
                                     n_samples: int = 500,
                                     n_operators: int = 2,
                                     sampling_strategy: str = 'random_without_replacement',
                                     optimization_method: str = 'exhaustive_discrete',
                                     use_latent_features: bool = True) -> Dict:
    """Complete workflow for discrete operator optimization"""
    
    print("=" * 60)
    print("DISCRETE OPERATOR OPTIMIZATION WORKFLOW")
    print("=" * 60)
    
    # Initialize optimizer
    optimizer = DiscreteOperatorOptimizer(
        codebook=codebook,
        simulator=simulator,
        n_operators=n_operators
    )
    
    # Step 1: Generate discrete samples
    start_time = time.time()
    combinations = optimizer.sample_operator_combinations(
        n_samples=n_samples,
        sampling_strategy=sampling_strategy
    )
    sampling_time = time.time() - start_time
    
    # Step 2: Evaluate combinations
    start_time = time.time()
    losses = optimizer.evaluate_combinations(
        combinations=combinations,
        x_val=x_val,
        y_val=y_val,
        model=model,
        batch_size=50
    )
    evaluation_time = time.time() - start_time
    
    # Step 3: Fit GP regressor
    start_time = time.time()
    gp_results = optimizer.fit_gp_regressor(combinations, losses, use_latent_features)
    fitting_time = time.time() - start_time
    
    # Step 4: Find optimal combination
    start_time = time.time()
    best_combination, best_loss, optimization_info = optimizer.find_optimal_combination(
        method=optimization_method
    )
    optimization_time = time.time() - start_time
    
    # Step 5: Visualize results
    fig = optimizer.visualize_results()
    
    # Compile results
    results = {
        'best_combination': best_combination,
        'best_predicted_loss': best_loss,
        'optimization_info': optimization_info,
        'gp_performance': gp_results,
        'timing': {
            'sampling': sampling_time,
            'evaluation': evaluation_time,
            'fitting': fitting_time,
            'optimization': optimization_time
        },
        'samples_info': {
            'n_samples': len(combinations),
            'n_valid': len([l for l in losses if np.isfinite(l)]),
            'loss_range': (min([l for l in losses if np.isfinite(l)] or [float('inf')]),
                          max([l for l in losses if np.isfinite(l)] or [0]))
        },
        'optimizer': optimizer,  # For further use
        'figure': fig
    }
    
    # Print summary
    print("\n" + "=" * 60)
    print("OPTIMIZATION RESULTS SUMMARY")
    print("=" * 60)
    print(f"Best combination found: {best_combination}")
    print(f"Predicted loss: {best_loss:.6f}")
    print(f"GP R² score: {gp_results['r2_score']:.4f}")
    print(f"Total time: {sum(results['timing'].values()):.2f}s")
    print(f"Valid samples: {results['samples_info']['n_valid']}/{results['samples_info']['n_samples']}")
    
    if 'top_10_combinations' in optimization_info:
        print("\nTop 5 combinations:")
        for i, combo_info in enumerate(optimization_info['top_10_combinations'][:5]):
            print(f"  {i+1}: {combo_info['combination']} -> {combo_info['predicted_loss']:.6f}")
    
    return results

In [ ]:
N_OUTPUT_FRAMES=50
dataloader = load_ood_dataset("E_ALL")
for batch in dataloader:
    test_input = batch["input"]
    test_target = batch["target"]
    break

In [ ]:
# Mock example - replace with your actual data

idx = 5
codebook = lit_model.codebook.clone().detach()

x = test_input[idx, -1].unsqueeze(0).cuda()
y = test_target[idx].unsqueeze(0).cuda()
#epochs=100
npreds = 1
#t = random.randint(0, test_input.shape[1]-npreds-1)
t = 10
npreds = 3
#t = 14

x_val = test_input[idx, t].cuda().unsqueeze(0)
y_val = test_input[idx, t+1:t+npreds+1].cuda().unsqueeze(0)

# Run optimization
results = run_discrete_operator_optimization(
    codebook=codebook,
    simulator=simulator,
    x_val=x_val,
    y_val=y_val,
    model=model,
    n_samples=1000,  # Sample 100 operator combinations
    n_operators=3,
    sampling_strategy='random_without_replacement',
    optimization_method='random_discrete', #'random_discrete', #'exhaustive_discrete',
    use_latent_features=True  # Use actual latent embeddings, not just indices
)

print(f"\nOptimal operators: {results['best_combination']}")
print(f"Predicted loss: {results['best_predicted_loss']:.6f}")

In [ ]:
#operators = [codebook[347], codebook[376]]

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def validate_predictions(results, codebook, multi_operator_splitting, x_val, y_val, model, top_k=10):
    """
    Validate GP predictions by computing true errors for top combinations
    
    Args:
        results: Output from your optimization
        codebook: Your operator codebook
        multi_operator_splitting: Your actual simulation function
        x_val, y_val: Your validation data
        model: Your trained model
        top_k: Number of top combinations to validate
    
    Returns:
        validation_results: Dict with true vs predicted errors
    """
    
    print("=" * 60)
    print("VALIDATING GP PREDICTIONS vs TRUE ERRORS")
    print("=" * 60)
    
    # Get model device
    device = next(model.parameters()).device
    
    # Get top combinations from optimization
    if 'top_10_combinations' in results['optimization_info']:
        top_combinations = results['optimization_info']['top_10_combinations'][:top_k]
    else:
        # Create from best combination if top_10 not available
        best_combo = results['best_combination']
        top_combinations = [{'combination': best_combo, 'predicted_loss': results['best_predicted_loss']}]
    
    validation_results = []
    
    print(f"Validating top {len(top_combinations)} combinations...")
    print("-" * 60)
    
    for i, combo_info in enumerate(top_combinations):
        combination = combo_info['combination']
        predicted_loss = combo_info['predicted_loss']
        
        print(f"\n{i+1}. Testing combination {combination}")
        print(f"   Predicted loss: {predicted_loss:.6f}")
        
        try:
            # Create operator embeddings
            operator_embeddings = []
            for op_idx in combination:
                embedding = torch.tensor(
                    codebook[op_idx:op_idx+1], 
                    dtype=torch.float32, 
                    device=device
                )
                operator_embeddings.append(embedding)
            
            # Prepare input
            if x_val.dim() == 2:
                x_input = x_val.unsqueeze(0).to(device)
            else:
                x_input = x_val.to(device)
            
            y_val_device = y_val.to(device)
            
            # Run actual simulation
            with torch.no_grad():
                pred = multi_operator_splitting(
                    model=model,
                    theta_latents=operator_embeddings,
                    x=x_input,
                    nt=y_val.shape[0] if y_val.dim() == 2 else y_val.shape[1],
                    dt=4/250,
                    refinement_factor=5,
                    splitting_method="strang"
                )
            
            # Handle shape alignment
            if pred.shape != y_val_device.shape:
                if pred.dim() == 3 and y_val_device.dim() == 2 and pred.shape[0] == 1:
                    pred = pred.squeeze(0)
                elif pred.dim() == 2 and y_val_device.dim() == 3 and y_val_device.shape[0] == 1:
                    y_val_device = y_val_device.squeeze(0)
            
            # Compute true loss
            true_loss = torch.mean((pred - y_val_device)**2).item()
            
            # Compute error metrics
            error_ratio = true_loss / predicted_loss if predicted_loss > 0 else float('inf')
            absolute_error = abs(true_loss - predicted_loss)
            relative_error = absolute_error / true_loss if true_loss > 0 else float('inf')
            
            validation_results.append({
                'rank': i + 1,
                'combination': combination,
                'predicted_loss': predicted_loss,
                'true_loss': true_loss,
                'error_ratio': error_ratio,
                'absolute_error': absolute_error,
                'relative_error': relative_error,
                'status': 'success'
            })
            
            print(f"   True loss:      {true_loss:.6f}")
            print(f"   Error ratio:    {error_ratio:.3f}x")
            print(f"   Absolute error: {absolute_error:.6f}")
            print(f"   Relative error: {relative_error*100:.2f}%")
            
            if error_ratio > 2.0:
                print("   ⚠️  Large prediction error!")
            elif error_ratio > 1.5:
                print("   ⚠️  Moderate prediction error")
            else:
                print("   ✅ Good prediction!")
            
        except Exception as e:
            print(f"   ❌ Validation failed: {e}")
            validation_results.append({
                'rank': i + 1,
                'combination': combination,
                'predicted_loss': predicted_loss,
                'true_loss': float('inf'),
                'error': str(e),
                'status': 'failed'
            })
    
    # Summary statistics
    successful_validations = [r for r in validation_results if r['status'] == 'success']
    
    if successful_validations:
        print("\n" + "=" * 60)
        print("VALIDATION SUMMARY")
        print("=" * 60)
        
        error_ratios = [r['error_ratio'] for r in successful_validations if np.isfinite(r['error_ratio'])]
        relative_errors = [r['relative_error'] for r in successful_validations if np.isfinite(r['relative_error'])]
        
        print(f"Successful validations: {len(successful_validations)}/{len(validation_results)}")
        
        if error_ratios:
            print(f"Error ratio statistics:")
            print(f"  Mean: {np.mean(error_ratios):.3f}x")
            print(f"  Median: {np.median(error_ratios):.3f}x")
            print(f"  Min: {np.min(error_ratios):.3f}x")
            print(f"  Max: {np.max(error_ratios):.3f}x")
        
        if relative_errors:
            print(f"Relative error statistics:")
            print(f"  Mean: {np.mean(relative_errors)*100:.2f}%")
            print(f"  Median: {np.median(relative_errors)*100:.2f}%")
        
        # Find the actual best combination
        true_best = min(successful_validations, key=lambda x: x['true_loss'])
        predicted_best = successful_validations[0]  # First in list (best predicted)
        
        print(f"\nActual best combination: {true_best['combination']} (true loss: {true_best['true_loss']:.6f})")
        print(f"GP predicted best: {predicted_best['combination']} (predicted loss: {predicted_best['predicted_loss']:.6f})")
        
        if true_best['combination'] == predicted_best['combination']:
            print("✅ GP correctly identified the best combination!")
        else:
            print("⚠️  GP predicted a different best combination")
            print(f"    Predicted best true loss: {predicted_best['true_loss']:.6f}")
            print(f"    Actual best rank in GP: {true_best['rank']}")
    
    return validation_results

def plot_prediction_accuracy(validation_results, save_path=None):
    """Plot predicted vs true losses"""
    
    successful = [r for r in validation_results if r['status'] == 'success']
    
    if not successful:
        print("No successful validations to plot")
        return None
    
    predicted = [r['predicted_loss'] for r in successful]
    true = [r['true_loss'] for r in successful]
    combinations = [str(r['combination']) for r in successful]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot
    ax1.scatter(predicted, true, alpha=0.7, s=100)
    
    # Perfect prediction line
    min_val = min(min(predicted), min(true))
    max_val = max(max(predicted), max(true))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
    
    # Add combination labels
    for i, combo in enumerate(combinations):
        ax1.annotate(f'{i+1}', (predicted[i], true[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax1.set_xlabel('Predicted Loss')
    ax1.set_ylabel('True Loss')
    ax1.set_title('GP Prediction Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Compute R²
    from sklearn.metrics import r2_score
    r2 = r2_score(true, predicted)
    ax1.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax1.transAxes,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Error ratios bar plot
    error_ratios = [r['error_ratio'] for r in successful if np.isfinite(r['error_ratio'])]
    ranks = [r['rank'] for r in successful if np.isfinite(r['error_ratio'])]
    
    colors = ['green' if ratio <= 1.5 else 'orange' if ratio <= 2.0 else 'red' for ratio in error_ratios]
    
    ax2.bar(range(len(error_ratios)), error_ratios, color=colors, alpha=0.7)
    ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Perfect Prediction')
    ax2.axhline(y=1.5, color='orange', linestyle='--', alpha=0.5, label='1.5x Error')
    ax2.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='2x Error')
    
    ax2.set_xlabel('Combination Rank')
    ax2.set_ylabel('Error Ratio (True/Predicted)')
    ax2.set_title('Prediction Error by Rank')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Set x-axis labels to show ranks
    ax2.set_xticks(range(len(error_ratios)))
    ax2.set_xticklabels([f'{r}' for r in ranks])
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    return fig

def quick_validation(results, codebook, multi_operator_splitting, x_val, y_val, model):
    """Quick validation of just the best combination"""
    
    print("QUICK VALIDATION - Best Combination Only")
    print("=" * 50)
    
    best_combo = results['best_combination']
    try:
        predicted_loss = results['best_predicted_loss']
    except:
        predicted_loss = results['predicted_loss']
        
    
    print(f"Best combination: {best_combo}")
    print(f"Predicted loss: {predicted_loss:.6f}")
    relative_l2_error = RelativeL2()
    
    # Get model device
    device = next(model.parameters()).device
    
    try:
        # Create operator embeddings
        operator_embeddings = []
        for op_idx in best_combo:
            embedding = torch.tensor(
                codebook[op_idx:op_idx+1], 
                dtype=torch.float32, 
                device=device
            )
            operator_embeddings.append(embedding)
        
        # Prepare input
        if x_val.dim() == 2:
            x_input = x_val.unsqueeze(0).to(device)
        else:
            x_input = x_val.to(device)
        
        y_val_device = y_val.to(device)
        
        # Run actual simulation
        with torch.no_grad():
            pred = multi_operator_splitting(
                model=model,
                theta_latents=operator_embeddings,
                x=x_input,
                nt=y_val.shape[0] if y_val.dim() == 2 else y_val.shape[1],
                dt=4/250,
                refinement_factor=5,
                splitting_method="strang"
            )
        
        # Handle shape alignment
        if pred.shape != y_val_device.shape:
            if pred.dim() == 3 and y_val_device.dim() == 2 and pred.shape[0] == 1:
                pred = pred.squeeze(0)
            elif pred.dim() == 2 and y_val_device.dim() == 3 and y_val_device.shape[0] == 1:
                y_val_device = y_val_device.squeeze(0)
        
        # Compute true loss
        #true_loss = torch.mean((pred - y_val_device)**2).item()
        true_loss = relative_l2_error(pred, y_val_device).mean()
        
        print(f"True loss: {true_loss:.6f}")
        print(f"Error ratio: {true_loss/predicted_loss:.3f}x")
        print(f"Absolute error: {abs(true_loss - predicted_loss):.6f}")
        
        if true_loss/predicted_loss <= 1.5:
            print("✅ Good prediction accuracy!")
        else:
            print("⚠️  Large prediction error - GP may need more training data")
        
        return {
            'predicted_loss': predicted_loss,
            'true_loss': true_loss,
            'error_ratio': true_loss/predicted_loss,
            'success': True,
            'pred': pred,
        }
        
    except Exception as e:
        print(f"❌ Validation failed: {e}")
        return {'success': False, 'error': str(e)}

In [ ]:
# After your optimization completes:

# Quick validation (just best combination)
validation = quick_validation(
    results=results,
    codebook=codebook,
    multi_operator_splitting=multi_operator_splitting,
    x_val=test_input[idx, -1].unsqueeze(0),
    y_val=test_target[idx, :].unsqueeze(0),
    model=model
)

# Full validation (top 10 combinations)
#validation_results = validate_predictions(
#    results=results,
#    codebook=codebook,
#    multi_operator_splitting=multi_operator_splitting,
#    x_val=x_val,
#    y_val=y_val,
#    model=model,
#    top_k=10
#)

# Plot accuracy
#fig = plot_prediction_accuracy(validation_results, save_path='prediction_accuracy.png')
#plt.show()

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(validation["pred"][:, t].cpu().detach().squeeze())

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(test_target[idx, t].cpu().detach().squeeze())

In [ ]:
!pip install xgboost

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import List, Tuple, Callable, Dict
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import itertools
import time

class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x).squeeze(-1)

class MinimalOperatorOptimizer:
    """Clean, minimal operator optimization with XGBoost or Neural Network"""
    
    def __init__(self, codebook, simulator, model_type='xgboost', n_operators=2):
        self.codebook = codebook.detach().cpu().numpy()
        self.n_codes = len(codebook)
        self.n_operators = n_operators
        self.simulator = simulator
        self.model_type = model_type
        
        self.regressor = None
        self.scaler = StandardScaler()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.relative_l2_error = RelativeL2()
        
        print(f"Initialized with {self.n_codes} operators using {model_type} for {n_operators} operator combinations")
        
    def sample_combinations(self, n_samples):
        """Sample random operator combinations for any number of operators"""
        combinations = []
        for _ in range(n_samples):
            combo = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
            combinations.append(combo)
        
        # Remove duplicates
        unique_combinations = []
        seen = set()
        for combo in combinations:
            combo_tuple = tuple(combo)
            if combo_tuple not in seen:
                seen.add(combo_tuple)
                unique_combinations.append(combo)
        
        print(f"Generated {len(unique_combinations)} unique combinations")
        return unique_combinations
    
    def evaluate_combinations(self, combinations, x_val, y_val, model, batch_size=200):
        """Evaluate combinations using simulator"""
        print(f"Evaluating {len(combinations)} combinations...")
        
        all_losses = []
        for i in range(0, len(combinations), batch_size):
            batch = combinations[i:i+batch_size]
            batch_losses = self._evaluate_batch(batch, x_val, y_val, model)
            all_losses.extend(batch_losses)
            
            if i % (batch_size * 10) == 0:
                print(f"  Processed {i + len(batch)}/{len(combinations)}")
        
        valid_losses = [loss for loss in all_losses if np.isfinite(loss)]
        print(f"Valid evaluations: {len(valid_losses)}/{len(all_losses)}")
        
        return all_losses
    
    def _evaluate_batch(self, combinations, x_val, y_val, model):
        """Evaluate a batch of combinations for any number of operators"""
        device = x_val.device
        batch_size = len(combinations)
        
        # Convert to embeddings for any number of operators
        operator_embeddings = []
        for op_pos in range(self.n_operators):
            indices = [combo[op_pos] for combo in combinations]
            embeddings = torch.tensor(self.codebook[indices], dtype=torch.float32, device=device)
            operator_embeddings.append(embeddings)
        
        x_batch = x_val.repeat(batch_size, 1, 1)
        
        try:
            with torch.no_grad():
                pred = self.simulator(model, operator_embeddings, x_batch,
                                    nt=y_val.shape[1], dt=4/250, refinement_factor=5,
                                    splitting_method="strang")
                
                # Compute relative L2 error (adjust based on your loss function)
                #losses = torch.mean((pred - y_val)**2, dim=(1, 2)).cpu().numpy()
                losses = self.relative_l2_error(pred, y_val).cpu().numpy()
            
            return losses.tolist()
        except Exception as e:
            print(f"Batch evaluation failed: {e}")
            return [float('inf')] * batch_size
    
    def prepare_features(self, combinations):
        """Convert combinations to feature vectors using embeddings for ANY number of operators"""
        features = []
        for combo in combinations:
            # Concatenate embeddings of ALL operators in the combination
            combined_embeddings = []
            for op_idx in combo:
                combined_embeddings.append(self.codebook[op_idx])
            combined = np.concatenate(combined_embeddings)
            features.append(combined)
        
        return np.array(features)
    
    def fit_model(self, combinations, losses):
        """Fit the selected model type"""
        print(f"Fitting {self.model_type} model...")
        
        # Filter valid samples
        valid_indices = [i for i, loss in enumerate(losses) if np.isfinite(loss)]
        valid_combinations = [combinations[i] for i in valid_indices]
        valid_losses = [losses[i] for i in valid_indices]
        
        if len(valid_indices) < 10:
            raise ValueError(f"Need at least 10 valid samples, got {len(valid_indices)}")
        
        # Prepare features
        X = self.prepare_features(valid_combinations)
        y = np.array(valid_losses)
        
        print(f"Feature dimension: {X.shape[1]} (concatenated {self.n_operators} operators × {X.shape[1]//self.n_operators}D embeddings)")
        
        if self.model_type == 'xgboost':
            self.regressor = xgb.XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                random_state=42
            )
            self.regressor.fit(X, y)
            
        elif self.model_type == 'neural':
            # Standardize features
            X_scaled = self.scaler.fit_transform(X)
            
            # Create neural network
            input_dim = X.shape[1]
            self.regressor = SimpleNeuralNet(input_dim).to(self.device)
            
            # Training
            self._train_neural_network(X_scaled, y)
        
        # Evaluate model
        if self.model_type == 'xgboost':
            y_pred = self.regressor.predict(X)
        else:  # neural
            X_scaled = self.scaler.transform(X) if self.model_type == 'neural' else X
            X_tensor = torch.FloatTensor(X_scaled).to(self.device)
            
            self.regressor.eval()
            with torch.no_grad():
                y_pred = self.regressor(X_tensor).cpu().numpy()
        
        r2 = r2_score(y, y_pred)
        print(f"Model R² score: {r2:.4f}")
        
        return r2
    
    def _train_neural_network(self, X_scaled, y, epochs=500):
        """Train neural network"""
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        y_tensor = torch.FloatTensor(y).to(self.device)
        
        optimizer = torch.optim.Adam(self.regressor.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        self.regressor.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            pred = self.regressor(X_tensor)
            loss = criterion(pred, y_tensor)
            loss.backward()
            optimizer.step()
            
            if epoch % 50 == 0:
                print(f"  Epoch {epoch}: Loss = {loss.item():.6f}")
    
    def find_best_combination(self, n_candidates=50000):
        """Find best combination using trained model for any number of operators"""
        print(f"Searching for best combination among {n_candidates} candidates...")
        
        # Generate candidate combinations for any number of operators
        candidates = []
        for _ in range(n_candidates):
            combo = [np.random.randint(0, self.n_codes) for _ in range(self.n_operators)]
            candidates.append(combo)
        
        # Remove duplicates
        unique_candidates = []
        seen = set()
        for combo in candidates:
            combo_tuple = tuple(combo)
            if combo_tuple not in seen:
                seen.add(combo_tuple)
                unique_candidates.append(combo)
        
        # Predict losses
        X_candidates = self.prepare_features(unique_candidates)
        
        if self.model_type == 'xgboost':
            predictions = self.regressor.predict(X_candidates)
        else:  # neural
            X_scaled = self.scaler.transform(X_candidates)
            X_tensor = torch.FloatTensor(X_scaled).to(self.device)
            
            self.regressor.eval()
            with torch.no_grad():
                predictions = self.regressor(X_tensor).cpu().numpy()
        
        # Find best
        best_idx = np.argmin(predictions)
        best_combination = unique_candidates[best_idx]
        best_predicted_loss = predictions[best_idx]
        
        print(f"Best combination: {best_combination}")
        print(f"Predicted loss: {best_predicted_loss:.6f}")
        
        return best_combination, best_predicted_loss

def optimize_operators(codebook, simulator, x_val, y_val, model, 
                      n_samples=1000, model_type='xgboost', n_operators=2):
    """
    Main function to optimize operator combinations
    
    Args:
        codebook: Tensor of operator embeddings [n_operators, embedding_dim]
        simulator: Function that runs multi_operator_splitting
        x_val, y_val: Validation data
        model: Trained neural network model
        n_samples: Number of combinations to sample and evaluate
        model_type: 'xgboost' or 'neural'
        n_operators: Number of operators in each combination
    
    Returns:
        Dictionary with optimization results
    """
    
    print("=" * 50)
    print(f"MINIMAL OPERATOR OPTIMIZATION ({model_type.upper()}) - {n_operators} operators")
    print("=" * 50)
    
    start_time = time.time()
    
    # Initialize optimizer
    optimizer = MinimalOperatorOptimizer(codebook, simulator, model_type, n_operators)
    
    # Sample combinations
    combinations = optimizer.sample_combinations(n_samples)
    
    # Evaluate combinations
    losses = optimizer.evaluate_combinations(combinations, x_val, y_val, model)
    
    # Fit model
    r2_score_val = optimizer.fit_model(combinations, losses)
    
    # Find best combination
    best_combination, best_loss = optimizer.find_best_combination()
    
    total_time = time.time() - start_time
    
    # Results
    results = {
        'best_combination': best_combination,
        'predicted_loss': best_loss,
        'model_r2': r2_score_val,
        'n_samples_evaluated': len([l for l in losses if np.isfinite(l)]),
        'total_time': total_time,
        'model_type': model_type,
        'n_operators': n_operators
    }
    
    print(f"\nOptimization complete in {total_time:.2f}s")
    print(f"Best operators: {best_combination}")
    print(f"Predicted loss: {best_loss:.6f}")
    print(f"Model accuracy: R² = {r2_score_val:.4f}")
    
    return results

In [ ]:
idx = 26
codebook = lit_model.codebook.clone().detach()

x = test_input[idx, -1].unsqueeze(0).cuda()
y = test_target[idx].unsqueeze(0).cuda()
#epochs=100
#npreds = 5
#t = random.randint(0, test_input.shape[1]-npreds-1)
t = 0
npreds = 15
#t = 14

x_val = test_input[idx, t].cuda().unsqueeze(0)
y_val = test_input[idx, t+1:t+npreds+1].cuda().unsqueeze(0)

In [ ]:
# Assuming you have:
# - your_codebook: [384, 3] tensor of trained operator embeddings
# - multi_operator_splitting: your simulation function
# - x_val: your initial condition
# - y_val: your target trajectory
# - your_model: your trained neural network

# Direct usage (if interface matches):
results = optimize_operators(
    codebook=codebook,
    simulator=multi_operator_splitting,
    x_val=x_val,
    y_val=y_val,
    model=model,
    n_samples=2000,  # More samples for better results
    model_type='neural',
    n_operators=3
)

In [ ]:
print(f"Best operator pair: {results['best_combination']}")
print(f"Predicted loss: {results['predicted_loss']:.6f}")
print(f"Model accuracy: R² = {results['model_r2']:.3f}")

# Get the actual operators from your codebook:
#best_op1_idx, best_op2_idx = results['best_combination']
#best_op1_embedding = codebook[best_op1_idx]
#best_op2_embedding = codebook[best_op2_idx]

#print(f"Best operator 1 embedding: {best_op1_embedding}")
#print(f"Best operator 2 embedding: {best_op2_embedding}")

In [ ]:
validation = quick_validation(
    results=results,
    codebook=codebook,
    multi_operator_splitting=multi_operator_splitting,
    x_val=test_input[idx, -1].unsqueeze(0),
    y_val=test_target[idx, :].unsqueeze(0),
    model=model
)

In [ ]:
validation

In [ ]:
pred = validation['pred']

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred[:, t].cpu().detach().squeeze())

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(test_target[idx, t].cpu().detach().squeeze())

In [ ]:
N_OUTPUT_FRAMES=50
BATCH_SIZE = 64
dataloader = load_ood_dataset("E_ALL")
for batch in dataloader:
    test_input = batch["input"]
    test_target = batch["target"]
    break

In [ ]:
test_input.shape

In [ ]:
all_losses = []
for idx in range(test_input.shape[0]):
    x = test_input[idx, -1].unsqueeze(0).cuda()
    y = test_target[idx].unsqueeze(0).cuda()
    t = 0
    npreds = 15

    x_val = test_input[idx, t].cuda().unsqueeze(0)
    y_val = test_input[idx, t+1:t+npreds+1].cuda().unsqueeze(0)

    results = optimize_operators(
        codebook=codebook,
        simulator=multi_operator_splitting,
        x_val=x_val,
        y_val=y_val,
        model=model,
        n_samples=200,  # More samples for better results
        model_type='neural',
        n_operators=3
    )
    
    validation = quick_validation(
    results=results,
    codebook=codebook,
    multi_operator_splitting=multi_operator_splitting,
    x_val=test_input[idx, -1].unsqueeze(0),
    y_val=test_target[idx, :].unsqueeze(0),
    model=model)

    all_losses.append(validation['true_loss'])
    